<a href="https://colab.research.google.com/github/Japr2206/RF/blob/main/DYNAMIC_STABILITY_INDEX_FOR_EARLY_LEARNING_RATE_SELECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# PROJECT:
# DYNAMIC STABILITY AND LEARNING-RATE SELECTION
# IN DEEP NEURAL NETWORKS
# MASTER NOTEBOOK — CLEAN, ORDERED, ROBUST VERSION
# ==========================================================

# ==========================================================
# BLOCK 0 — DRIVE MOUNT AND PERSISTENT PATHS
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import json
import time
import math
import random
import pickle
import itertools
from dataclasses import dataclass, asdict
from typing import Dict, List, Callable, Tuple, Any, Optional

PROJECT_ROOT = "/content/drive/MyDrive/dynamic_stability_lr_project"
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "paper_outputs")
DATA_ROOT = os.path.join(PROJECT_ROOT, "data_cache")

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print("Drive mounted successfully.")
print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)

# ==========================================================
# BLOCK 1 — LIBRARIES
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, Subset, random_split, TensorDataset

from sklearn.datasets import load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_rel, wilcoxon

# ==========================================================
# BLOCK 2 — GLOBAL CONFIGURATION
# ==========================================================

@dataclass
class ExperimentConfig:
    seed: int = 42

    data_dir: str = DATA_ROOT
    output_dir: str = OUTPUT_ROOT

    batch_size_train: int = 128
    batch_size_eval: int = 256
    num_workers: int = 2
    pin_memory: bool = True

    lr_min: float = 1e-4
    lr_max: float = 1.0
    lr_points: int = 12

    mnist_subset: int = 10000
    fashion_subset: int = 8000
    cifar10_subset: int = 10000
    cifar100_subset: int = 10000
    multidataset_vision_subset: int = 6000

    exploratory_epochs: int = 2
    main_epochs: int = 5
    final_epochs_small: int = 10
    final_epochs_cifar_cnn: int = 20
    final_epochs_cifar_resnet: int = 24

    dynamic_search_max_batches: int = 30
    early_loss_max_batches: int = 16
    random_search_samples: int = 4

    seeds_multirun: Tuple[int, ...] = (7, 21, 42, 84, 123)

    eps: float = 1e-8
    min_series_points: int = 8

    dsi_alpha_volatility: float = 1.0
    dsi_beta_oscillation: float = 0.75
    dsi_gamma_descent: float = 1.25
    dsi_delta_instability: float = 2.0

    dynamic_quantiles: Tuple[float, ...] = (0.25, 0.33, 0.50)

    alpha_grid: Tuple[float, ...] = (0.5, 1.0, 1.5)
    beta_grid: Tuple[float, ...] = (0.5, 0.75, 1.0)
    gamma_grid: Tuple[float, ...] = (1.0, 1.25, 1.5)
    delta_grid: Tuple[float, ...] = (1.0, 2.0, 3.0)

    run_cifar100: bool = True
    run_multiseed: bool = True
    run_scheduler_baselines: bool = True
    run_hybrid_method: bool = True
    run_selection_sensitivity: bool = True
    run_dsi_weight_sensitivity: bool = True

CONFIG = ExperimentConfig()

os.makedirs(CONFIG.data_dir, exist_ok=True)
os.makedirs(CONFIG.output_dir, exist_ok=True)

print("CONFIG.data_dir   =", CONFIG.data_dir)
print("CONFIG.output_dir =", CONFIG.output_dir)

# ==========================================================
# BLOCK 3 — REPRODUCIBILITY AND DEVICE
# ==========================================================

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_global_seed(CONFIG.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Global seed:", CONFIG.seed)
print("Device:", DEVICE)

LEARNING_RATES = np.logspace(
    np.log10(CONFIG.lr_min),
    np.log10(CONFIG.lr_max),
    CONFIG.lr_points
)

print("Global learning-rate grid:")
print(LEARNING_RATES)

with open(os.path.join(CONFIG.output_dir, "experiment_config.json"), "w", encoding="utf-8") as f:
    json.dump(asdict(CONFIG), f, indent=4)

# ==========================================================
# BLOCK 4 — GENERAL UTILITIES
# ==========================================================

def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s/]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text

def make_loader(dataset, batch_size: int, shuffle: bool, seed: Optional[int] = None) -> DataLoader:
    generator = None
    if shuffle and seed is not None:
        generator = torch.Generator().manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CONFIG.num_workers,
        pin_memory=CONFIG.pin_memory if DEVICE.type == "cuda" else False,
        worker_init_fn=seed_worker,
        generator=generator
    )
    loader._experiment_shuffle = shuffle
    return loader

def rebuild_loader_with_seed(loader: DataLoader, seed: int) -> DataLoader:
    shuffle = getattr(loader, "_experiment_shuffle", False)
    generator = torch.Generator().manual_seed(seed) if shuffle else None

    cloned_loader = DataLoader(
        loader.dataset,
        batch_size=loader.batch_size,
        shuffle=shuffle,
        num_workers=loader.num_workers,
        pin_memory=loader.pin_memory,
        drop_last=loader.drop_last,
        collate_fn=loader.collate_fn,
        worker_init_fn=seed_worker,
        generator=generator
    )
    cloned_loader._experiment_shuffle = shuffle
    return cloned_loader

def save_pickle(obj: Any, filename: str) -> None:
    path = os.path.join(CONFIG.output_dir, filename)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def save_json(obj: Any, filename: str) -> None:
    path = os.path.join(CONFIG.output_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False)

def save_html_text(html_content: str, filename: str) -> None:
    path = os.path.join(CONFIG.output_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_content)

def export_dataframe_html(df: pd.DataFrame, title: str, filename: str) -> None:
    html_content = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>{title}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: white;
                color: black;
            }}
            h2 {{
                text-align: center;
            }}
            table {{
                border-collapse: collapse;
                width: 96%;
                margin: auto;
            }}
            th, td {{
                border: 1px solid #999;
                padding: 8px 10px;
                text-align: center;
            }}
            th {{
                background-color: #f2f2f2;
            }}
        </style>
    </head>
    <body>
        <h2>{title}</h2>
        {df.to_html(index=False)}
    </body>
    </html>
    """
    save_html_text(html_content, filename)

def save_table(df: pd.DataFrame, stem: str, title: str) -> None:
    df.to_csv(os.path.join(CONFIG.output_dir, f"{stem}.csv"), index=False)
    export_dataframe_html(df, title, f"{stem}.html")

def sanitize_series(values: List[float]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return arr

def has_non_finite(values: List[float]) -> bool:
    arr = np.asarray(values, dtype=float)
    return not np.all(np.isfinite(arr))

def safe_mean(values: List[float]) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) > 0 else np.nan

def safe_divide(numerator: float, denominator: float) -> float:
    if numerator is None or denominator is None:
        return np.nan
    if not np.isfinite(numerator) or not np.isfinite(denominator):
        return np.nan
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)

def zscore_safe(series: pd.Series) -> pd.Series:
    mean_val = series.mean()
    std_val = series.std()

    if std_val is None or np.isnan(std_val) or std_val == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return (series - mean_val) / std_val

def random_subset(dataset, subset_size: int, seed: int) -> Subset:
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(subset_size, len(dataset)), replace=False)
    return Subset(dataset, indices.tolist())

def cohen_d_paired(x: np.ndarray, y: np.ndarray) -> float:
    diff = x - y
    std_diff = np.std(diff, ddof=1)
    if std_diff == 0 or np.isnan(std_diff):
        return np.nan
    return float(np.mean(diff) / std_diff)

# ==========================================================
# BLOCK 5 — MODEL DEFINITIONS
# ==========================================================

class BaseMLPImage(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Tuple[int, ...], output_dim: int):
        super().__init__()
        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if x.ndim > 2:
            x = x.view(x.size(0), -1)
        return self.net(x)

class VariableDepthMLP(nn.Module):
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256, depth: int = 3, output_dim: int = 10):
        super().__init__()
        layers = []
        current_dim = input_dim

        for _ in range(depth):
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if x.ndim > 2:
            x = x.view(x.size(0), -1)
        return self.net(x)

class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int, n_classes: int, image_size: int):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        reduced_size = image_size // 4
        flat_dim = 64 * reduced_size * reduced_size

        self.classifier = nn.Sequential(
            nn.Linear(flat_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.block(x)
        out = out + residual
        return self.relu(out)

class SmallResNetLike(nn.Module):
    def __init__(self, in_channels: int = 3, n_classes: int = 10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.layer1 = ResidualBlock(32)
        self.pool1 = nn.MaxPool2d(2)

        self.layer2_conv = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.layer2_res = ResidualBlock(64)
        self.pool2 = nn.MaxPool2d(2)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2_conv(x)
        x = self.layer2_res(x)
        x = self.pool2(x)
        x = self.classifier(x)
        return x

class TabularMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================================
# BLOCK 6 — MODEL BUILDERS
# ==========================================================

def build_mnist_mlp() -> nn.Module:
    return BaseMLPImage(784, (512, 256, 128, 64), 10)

def build_simple_image_mlp() -> nn.Module:
    return BaseMLPImage(784, (256, 128), 10)

def build_cifar_mlp(output_dim: int = 10) -> nn.Module:
    return BaseMLPImage(3072, (1024, 512, 256, 128), output_dim)

def build_mnist_cnn() -> nn.Module:
    return SimpleCNN(1, 10, 28)

def build_fashion_cnn() -> nn.Module:
    return SimpleCNN(1, 10, 28)

def build_cifar10_cnn() -> nn.Module:
    return SimpleCNN(3, 10, 32)

def build_cifar100_cnn() -> nn.Module:
    return SimpleCNN(3, 100, 32)

def build_cifar10_resnet_like() -> nn.Module:
    return SmallResNetLike(3, 10)

def build_cifar100_resnet_like() -> nn.Module:
    return SmallResNetLike(3, 100)

# ==========================================================
# BLOCK 7 — OPTIMIZERS AND SCHEDULERS
# ==========================================================

def build_optimizer(optimizer_name: str, params, lr: float):
    optimizer_name = optimizer_name.lower()

    if optimizer_name == "sgd":
        return optim.SGD(params, lr=lr)
    elif optimizer_name == "adam":
        return optim.Adam(params, lr=lr)
    elif optimizer_name == "rmsprop":
        return optim.RMSprop(params, lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

def build_scheduler(scheduler_name: str, optimizer, total_epochs: int):
    scheduler_name = scheduler_name.lower()

    if scheduler_name == "none":
        return None
    elif scheduler_name == "steplr":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=max(1, total_epochs // 2),
            gamma=0.5
        )
    elif scheduler_name == "cosineannealinglr":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=total_epochs
        )
    else:
        raise ValueError(f"Unsupported scheduler: {scheduler_name}")

# ==========================================================
# BLOCK 8 — GLOBAL NORMS AND CHECKS
# ==========================================================

def compute_global_weight_norm(model: nn.Module) -> float:
    sq_sum = 0.0
    for p in model.parameters():
        sq_sum += p.data.pow(2).sum().item()
    return math.sqrt(sq_sum)

def compute_global_grad_norm(model: nn.Module) -> float:
    sq_sum = 0.0
    for p in model.parameters():
        if p.grad is not None:
            sq_sum += p.grad.pow(2).sum().item()
    return math.sqrt(sq_sum)

def model_has_non_finite_params(model: nn.Module) -> bool:
    for p in model.parameters():
        if not torch.isfinite(p.data).all():
            return True
    return False

def model_has_non_finite_grads(model: nn.Module) -> bool:
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            return True
    return False

# ==========================================================
# BLOCK 9 — GENERAL EVALUATION
# ==========================================================

def evaluate_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total if total > 0 else np.nan

# ==========================================================
# BLOCK 10 — TRAINING WITH TRAJECTORY RECORDING
# ==========================================================

def run_training_trajectory(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    lr: float,
    optimizer_name: str = "sgd",
    epochs: int = 3,
    record_dynamics: bool = True,
    scheduler_name: str = "none",
    max_batches: Optional[int] = None,
    run_seed: Optional[int] = None
) -> Dict[str, Any]:
    if run_seed is not None:
        set_global_seed(run_seed)
        effective_train_loader = rebuild_loader_with_seed(train_loader, run_seed)
    else:
        effective_train_loader = train_loader

    model = model_builder().to(DEVICE)
    optimizer = build_optimizer(optimizer_name, model.parameters(), lr)
    scheduler = build_scheduler(scheduler_name, optimizer, epochs)
    criterion = nn.CrossEntropyLoss()

    loss_series = []
    grad_norm_series = []
    weight_norm_series = []

    numerical_instability = False
    diverged = False
    batches_processed = 0

    start_time = time.perf_counter()

    model.train()

    for epoch in range(epochs):
        for x, y in effective_train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)

            if not torch.isfinite(loss):
                numerical_instability = True
                diverged = True
                break

            loss.backward()

            if model_has_non_finite_grads(model):
                numerical_instability = True
                diverged = True
                break

            if record_dynamics:
                grad_norm_series.append(compute_global_grad_norm(model))

            optimizer.step()
            batches_processed += 1

            if model_has_non_finite_params(model):
                numerical_instability = True
                diverged = True
                break

            if record_dynamics:
                weight_norm_series.append(compute_global_weight_norm(model))
                loss_series.append(loss.item())

            if max_batches is not None and batches_processed >= max_batches:
                break

        if diverged:
            break

        if scheduler is not None:
            scheduler.step()

        if max_batches is not None and batches_processed >= max_batches:
            break

    end_time = time.perf_counter()

    return {
        "model": model,
        "loss_series": loss_series,
        "grad_norm_series": grad_norm_series,
        "weight_norm_series": weight_norm_series,
        "has_non_finite": numerical_instability or has_non_finite(loss_series),
        "diverged": diverged,
        "batches_processed": batches_processed,
        "elapsed_seconds": end_time - start_time
    }

# ==========================================================
# BLOCK 11 — DSI DEFINITION
# ==========================================================

def empirical_loss_volatility(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)

    if len(clean) < CONFIG.min_series_points:
        return np.nan

    rel_diffs = np.abs(np.diff(clean)) / (np.abs(clean[:-1]) + eps)

    if len(rel_diffs) == 0:
        return np.nan

    return float(np.mean(np.log1p(rel_diffs)))

def empirical_loss_oscillation(losses: List[float]) -> float:
    clean = sanitize_series(losses)

    if len(clean) < CONFIG.min_series_points:
        return np.nan

    deltas = np.diff(clean)

    if len(deltas) < 2:
        return np.nan

    sign_changes = (deltas[1:] * deltas[:-1] < 0).astype(float)

    if len(sign_changes) == 0:
        return np.nan

    return float(np.mean(sign_changes))

def normalized_loss_descent(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)

    if len(clean) < 2:
        return np.nan

    initial = clean[0]
    final = clean[-1]

    return float((initial - final) / (abs(initial) + eps))

def instability_penalty(losses: List[float], has_nonfinite_flag: bool = False) -> float:
    if has_nonfinite_flag or has_non_finite(losses):
        return 1.0
    return 0.0

def dynamic_stability_index(
    losses: List[float],
    has_nonfinite_flag: bool = False,
    alpha: float = CONFIG.dsi_alpha_volatility,
    beta: float = CONFIG.dsi_beta_oscillation,
    gamma: float = CONFIG.dsi_gamma_descent,
    delta: float = CONFIG.dsi_delta_instability
) -> Dict[str, float]:
    volatility = empirical_loss_volatility(losses)
    oscillation = empirical_loss_oscillation(losses)
    descent = normalized_loss_descent(losses)
    penalty = instability_penalty(losses, has_nonfinite_flag)

    if penalty > 0:
        total_dsi = np.inf
    elif np.isnan(volatility) or np.isnan(oscillation) or np.isnan(descent):
        total_dsi = np.nan
    else:
        total_dsi = alpha * volatility + beta * oscillation - gamma * descent + delta * penalty

    return {
        "dsi": float(total_dsi) if np.isfinite(total_dsi) else total_dsi,
        "volatility": volatility,
        "oscillation": oscillation,
        "descent": descent,
        "instability_penalty": penalty
    }

# ==========================================================
# BLOCK 12 — ORIGINAL EXPLORATORY METRIC
# ==========================================================

def original_log_diff_proxy(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)

    if len(clean) < CONFIG.min_series_points:
        return np.nan

    diffs = np.abs(np.diff(clean))
    diffs = diffs[diffs > eps]

    if len(diffs) == 0:
        return np.nan

    return float(np.mean(np.log(diffs)))

# ==========================================================
# BLOCK 13 — DSI ABLATION
# ==========================================================

def dsi_component_ablation(
    losses: List[float],
    has_nonfinite_flag: bool = False,
    alpha: float = CONFIG.dsi_alpha_volatility,
    beta: float = CONFIG.dsi_beta_oscillation,
    gamma: float = CONFIG.dsi_gamma_descent,
    delta: float = CONFIG.dsi_delta_instability
) -> Dict[str, float]:
    volatility = empirical_loss_volatility(losses)
    oscillation = empirical_loss_oscillation(losses)
    descent = normalized_loss_descent(losses)
    penalty = instability_penalty(losses, has_nonfinite_flag)

    full = dynamic_stability_index(
        losses,
        has_nonfinite_flag=has_nonfinite_flag,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        delta=delta
    )["dsi"]

    return {
        "full_dsi": full,
        "only_volatility": volatility,
        "only_oscillation": oscillation,
        "only_descent_reward": (-descent if not np.isnan(descent) else np.nan),
        "volatility_plus_oscillation": (
            volatility + oscillation if not (np.isnan(volatility) or np.isnan(oscillation)) else np.nan
        ),
        "penalty": penalty
    }

# ==========================================================
# BLOCK 14 — DSI WEIGHT SENSITIVITY
# ==========================================================

def run_dsi_weight_sensitivity(
    trajectories: Dict[float, Dict[str, Any]],
    rule_name: str = "quantile_plus_descent",
    quantile: float = 0.33
) -> pd.DataFrame:
    rows = []

    weight_combinations = list(itertools.product(
        CONFIG.alpha_grid,
        CONFIG.beta_grid,
        CONFIG.gamma_grid,
        CONFIG.delta_grid
    ))

    for alpha, beta, gamma, delta in weight_combinations:
        recalculated_rows = []

        for lr, traj in trajectories.items():
            loss_series = traj["loss_series"]
            dsi_dict = dynamic_stability_index(
                losses=loss_series,
                has_nonfinite_flag=traj["has_non_finite"],
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                delta=delta
            )

            recalculated_rows.append({
                "learning_rate": lr,
                "dsi": dsi_dict["dsi"],
                "descent": dsi_dict["descent"]
            })

        df_recalc = pd.DataFrame(recalculated_rows)

        try:
            selected_lr, configs_considered = select_lr_by_dynamic_pure(
                df_dynamic=df_recalc,
                rule_name=rule_name,
                quantile=quantile
            )
        except Exception:
            selected_lr, configs_considered = np.nan, np.nan

        rows.append({
            "alpha": alpha,
            "beta": beta,
            "gamma": gamma,
            "delta": delta,
            "rule_name": rule_name,
            "quantile": quantile,
            "selected_lr": selected_lr,
            "configs_considered": configs_considered
        })

    return pd.DataFrame(rows)

# ==========================================================
# BLOCK 15 — VISION DATASETS
# ==========================================================

def get_mnist_loaders(subset_size: int = CONFIG.mnist_subset, seed: int = CONFIG.seed):
    transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = torchvision.datasets.MNIST(
        root=CONFIG.data_dir, train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.MNIST(
        root=CONFIG.data_dir, train=False, download=True, transform=transform
    )

    train_subset = random_subset(train_dataset, subset_size, seed)

    n_total = len(train_subset)
    n_train = int(0.8 * n_total)
    n_val = n_total - n_train
    generator = torch.Generator().manual_seed(seed)

    train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

    train_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=seed)
    val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=seed)
    test_loader = make_loader(test_dataset, CONFIG.batch_size_eval, False, seed=seed)
    train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=seed)

    return train_loader, val_loader, test_loader, train_full_loader

def get_fashion_mnist_loaders(subset_size: int = CONFIG.fashion_subset, seed: int = CONFIG.seed):
    transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = torchvision.datasets.FashionMNIST(
        root=CONFIG.data_dir, train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.FashionMNIST(
        root=CONFIG.data_dir, train=False, download=True, transform=transform
    )

    train_subset = random_subset(train_dataset, subset_size, seed)

    n_total = len(train_subset)
    n_train = int(0.8 * n_total)
    n_val = n_total - n_train
    generator = torch.Generator().manual_seed(seed)

    train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

    train_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=seed)
    val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=seed)
    test_loader = make_loader(test_dataset, CONFIG.batch_size_eval, False, seed=seed)
    train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=seed)

    return train_loader, val_loader, test_loader, train_full_loader

def get_cifar10_loaders(subset_size: int = CONFIG.cifar10_subset, seed: int = CONFIG.seed):
    transform_train = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616))
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root=CONFIG.data_dir, train=True, download=True, transform=transform_train
    )
    test_dataset = torchvision.datasets.CIFAR10(
        root=CONFIG.data_dir, train=False, download=True, transform=transform_test
    )

    train_subset = random_subset(train_dataset, subset_size, seed)

    n_total = len(train_subset)
    n_train = int(0.8 * n_total)
    n_val = n_total - n_train
    generator = torch.Generator().manual_seed(seed)

    train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

    train_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=seed)
    val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=seed)
    test_loader = make_loader(test_dataset, CONFIG.batch_size_eval, False, seed=seed)
    train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=seed)

    return train_loader, val_loader, test_loader, train_full_loader

def get_cifar100_loaders(subset_size: int = CONFIG.cifar100_subset, seed: int = CONFIG.seed):
    transform_train = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761))
    ])

    train_dataset = torchvision.datasets.CIFAR100(
        root=CONFIG.data_dir, train=True, download=True, transform=transform_train
    )
    test_dataset = torchvision.datasets.CIFAR100(
        root=CONFIG.data_dir, train=False, download=True, transform=transform_test
    )

    train_subset = random_subset(train_dataset, subset_size, seed)

    n_total = len(train_subset)
    n_train = int(0.8 * n_total)
    n_val = n_total - n_train
    generator = torch.Generator().manual_seed(seed)

    train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

    train_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=seed)
    val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=seed)
    test_loader = make_loader(test_dataset, CONFIG.batch_size_eval, False, seed=seed)
    train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=seed)

    return train_loader, val_loader, test_loader, train_full_loader

# ==========================================================
# BLOCK 16 — TABULAR DATASETS
# ==========================================================

def get_tabular_loaders(dataset_name: str, seed: int = CONFIG.seed):
    dataset_name = dataset_name.lower()

    if dataset_name == "wine":
        data = load_wine()
    elif dataset_name == "breastcancer":
        data = load_breast_cancer()
    elif dataset_name == "digits":
        data = load_digits()
    else:
        raise ValueError(f"Unsupported tabular dataset: {dataset_name}")

    X = data.data
    y = data.target

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=seed, stratify=y_train_full
    )

    scaler_search = StandardScaler()
    X_train_scaled = scaler_search.fit_transform(X_train)
    X_val_scaled = scaler_search.transform(X_val)

    scaler_final = StandardScaler()
    X_train_full_scaled = scaler_final.fit_transform(X_train_full)
    X_test_scaled = scaler_final.transform(X_test)

    X_train_scaled = torch.tensor(X_train_scaled).float()
    X_val_scaled = torch.tensor(X_val_scaled).float()
    X_train_full_scaled = torch.tensor(X_train_full_scaled).float()
    X_test_scaled = torch.tensor(X_test_scaled).float()

    y_train = torch.tensor(y_train).long()
    y_val = torch.tensor(y_val).long()
    y_train_full = torch.tensor(y_train_full).long()
    y_test = torch.tensor(y_test).long()

    train_search_ds = TensorDataset(X_train_scaled, y_train)
    val_ds = TensorDataset(X_val_scaled, y_val)
    train_full_ds = TensorDataset(X_train_full_scaled, y_train_full)
    test_ds = TensorDataset(X_test_scaled, y_test)

    train_search_loader = make_loader(train_search_ds, 32, True, seed=seed)
    val_loader = make_loader(val_ds, 64, False, seed=seed)
    train_full_loader = make_loader(train_full_ds, 32, True, seed=seed)
    test_loader = make_loader(test_ds, 64, False, seed=seed)

    metadata = {
        "input_dim": X.shape[1],
        "n_classes": len(np.unique(y))
    }

    return train_search_loader, val_loader, test_loader, train_full_loader, metadata

# ==========================================================
# BLOCK 17 — GENERAL LR SWEEP
# ==========================================================

def run_lr_sweep(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    learning_rates: np.ndarray,
    optimizer_name: str = "sgd",
    epochs: int = 3,
    scheduler_name: str = "none",
    max_batches: Optional[int] = None,
    dsi_params: Optional[Dict[str, float]] = None,
    run_seed: Optional[int] = None
) -> Tuple[pd.DataFrame, Dict[float, Dict[str, Any]]]:
    if dsi_params is None:
        dsi_params = {}

    summary_rows = []
    trajectories = {}

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            optimizer_name=optimizer_name,
            epochs=epochs,
            record_dynamics=True,
            scheduler_name=scheduler_name,
            max_batches=max_batches,
            run_seed=run_seed
        )

        loss_series = out["loss_series"]

        dsi_dict = dynamic_stability_index(
            losses=loss_series,
            has_nonfinite_flag=out["has_non_finite"],
            **dsi_params
        )

        original_proxy = original_log_diff_proxy(loss_series)
        ablation_dict = dsi_component_ablation(
            loss_series,
            has_nonfinite_flag=out["has_non_finite"],
            **dsi_params
        )

        summary_rows.append({
            "learning_rate": float(lr),
            "optimizer": optimizer_name,
            "scheduler": scheduler_name,
            "dsi": dsi_dict["dsi"],
            "volatility": dsi_dict["volatility"],
            "oscillation": dsi_dict["oscillation"],
            "descent": dsi_dict["descent"],
            "instability_penalty": dsi_dict["instability_penalty"],
            "original_proxy": original_proxy,
            "ablation_only_volatility": ablation_dict["only_volatility"],
            "ablation_only_oscillation": ablation_dict["only_oscillation"],
            "ablation_only_descent_reward": ablation_dict["only_descent_reward"],
            "ablation_volatility_plus_oscillation": ablation_dict["volatility_plus_oscillation"],
            "final_loss": loss_series[-1] if len(loss_series) > 0 else np.nan,
            "n_steps": len(loss_series),
            "has_non_finite": out["has_non_finite"],
            "diverged": out["diverged"],
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"]
        })

        trajectories[float(lr)] = {
            "loss_series": out["loss_series"],
            "grad_norm_series": out["grad_norm_series"],
            "weight_norm_series": out["weight_norm_series"],
            "has_non_finite": out["has_non_finite"],
            "diverged": out["diverged"]
        }

    df_summary = pd.DataFrame(summary_rows).sort_values("learning_rate").reset_index(drop=True)
    return df_summary, trajectories

# ==========================================================
# BLOCK 18 — VALIDATION ACCURACY VS LR
# ==========================================================

def run_validation_accuracy_sweep(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rates: np.ndarray,
    optimizer_name: str = "sgd",
    epochs: int = 3,
    scheduler_name: str = "none",
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    rows = []

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            optimizer_name=optimizer_name,
            epochs=epochs,
            record_dynamics=False,
            scheduler_name=scheduler_name,
            run_seed=run_seed
        )

        if out["diverged"]:
            val_acc = np.nan
        else:
            val_acc = evaluate_accuracy(out["model"], val_loader)

        rows.append({
            "learning_rate": float(lr),
            "optimizer": optimizer_name,
            "scheduler": scheduler_name,
            "val_accuracy": val_acc,
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"],
            "diverged": out["diverged"]
        })

    return pd.DataFrame(rows).sort_values("learning_rate").reset_index(drop=True)

# ==========================================================
# BLOCK 19 — BASELINE: EARLY-LOSS SWEEP
# ==========================================================

def run_early_loss_sweep_baseline(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    learning_rates: np.ndarray,
    optimizer_name: str = "sgd",
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    rows = []

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            optimizer_name=optimizer_name,
            epochs=1,
            record_dynamics=True,
            scheduler_name="none",
            max_batches=CONFIG.early_loss_max_batches,
            run_seed=run_seed
        )

        loss_series = out["loss_series"]
        early_loss_mean = np.nan if out["diverged"] else safe_mean(loss_series)

        rows.append({
            "learning_rate": float(lr),
            "optimizer": optimizer_name,
            "early_loss_mean": early_loss_mean,
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"],
            "diverged": out["diverged"]
        })

    return pd.DataFrame(rows).sort_values("learning_rate").reset_index(drop=True)

def select_lr_by_early_loss_sweep(df_baseline: pd.DataFrame) -> float:
    df_valid = df_baseline.dropna(subset=["early_loss_mean"]).copy()
    if len(df_valid) == 0:
        raise ValueError("No valid configurations for early-loss sweep.")
    return float(df_valid.loc[df_valid["early_loss_mean"].idxmin(), "learning_rate"])

# ==========================================================
# BLOCK 20 — BASELINE: RANDOM SEARCH
# ==========================================================

def run_random_search_validation(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rates: np.ndarray,
    optimizer_name: str = "sgd",
    epochs: int = CONFIG.exploratory_epochs,
    n_samples: int = CONFIG.random_search_samples,
    sampling_seed: int = CONFIG.seed,
    run_seed: Optional[int] = None
) -> Tuple[float, pd.DataFrame]:
    unique_lrs = np.asarray(sorted({float(x) for x in learning_rates}))
    rng = np.random.default_rng(sampling_seed)
    sampled_lrs = np.sort(
        rng.choice(unique_lrs, size=min(n_samples, len(unique_lrs)), replace=False)
    )

    df_random = run_validation_accuracy_sweep(
        model_builder=model_builder,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rates=sampled_lrs,
        optimizer_name=optimizer_name,
        epochs=epochs,
        scheduler_name="none",
        run_seed=run_seed
    )

    best_lr = select_lr_by_grid_search(df_random)
    return best_lr, df_random

# ==========================================================
# BLOCK 21 — SELECTION RULES
# ==========================================================

def select_lr_by_grid_search(df_val: pd.DataFrame) -> float:
    df_valid = df_val.dropna(subset=["val_accuracy"]).copy()
    if len(df_valid) == 0:
        raise ValueError("No valid configurations for grid search.")
    return float(df_valid.loc[df_valid["val_accuracy"].idxmax(), "learning_rate"])

def dynamic_rule_min_dsi(df_dynamic: pd.DataFrame) -> Tuple[float, int]:
    df_valid = df_dynamic[df_dynamic["dsi"].notna()].copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]
    if len(df_valid) == 0:
        raise ValueError("No valid configurations for min_dsi.")
    best_row = df_valid.loc[df_valid["dsi"].idxmin()]
    return float(best_row["learning_rate"]), 1

def dynamic_rule_quantile_plus_descent(df_dynamic: pd.DataFrame, quantile: float) -> Tuple[float, int]:
    df_valid = df_dynamic.dropna(subset=["dsi", "descent"]).copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]

    if len(df_valid) == 0:
        raise ValueError("No valid configurations for quantile_plus_descent.")

    threshold = df_valid["dsi"].quantile(quantile)
    df_region = df_valid[df_valid["dsi"] <= threshold].copy()

    if len(df_region) == 0:
        best_row = df_valid.loc[df_valid["dsi"].idxmin()]
        return float(best_row["learning_rate"]), 1

    best_row = df_region.loc[df_region["descent"].idxmax()]
    return float(best_row["learning_rate"]), len(df_region)

def dynamic_rule_rank_combined(df_dynamic: pd.DataFrame) -> Tuple[float, int]:
    df_valid = df_dynamic.dropna(subset=["dsi", "descent"]).copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]

    if len(df_valid) == 0:
        raise ValueError("No valid configurations for rank_combined.")

    df_valid["rank_dsi"] = df_valid["dsi"].rank(method="average", ascending=True)
    df_valid["rank_descent"] = df_valid["descent"].rank(method="average", ascending=False)
    df_valid["combined_rank_score"] = -(df_valid["rank_dsi"] + df_valid["rank_descent"])

    best_row = df_valid.loc[df_valid["combined_rank_score"].idxmax()]
    return float(best_row["learning_rate"]), len(df_valid)

def select_lr_by_dynamic_pure(
    df_dynamic: pd.DataFrame,
    rule_name: str = "quantile_plus_descent",
    quantile: float = 0.33
) -> Tuple[float, int]:
    rule_name = rule_name.lower()

    if rule_name == "min_dsi":
        return dynamic_rule_min_dsi(df_dynamic)
    elif rule_name == "quantile_plus_descent":
        return dynamic_rule_quantile_plus_descent(df_dynamic, quantile)
    elif rule_name == "rank_combined":
        return dynamic_rule_rank_combined(df_dynamic)
    else:
        raise ValueError(f"Unsupported dynamic rule: {rule_name}")

def select_lr_by_hybrid_score(df_merged: pd.DataFrame) -> Tuple[float, int]:
    df_valid = df_merged.dropna(subset=["dsi", "val_accuracy"]).copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]

    if len(df_valid) == 0:
        raise ValueError("No valid configurations for the hybrid criterion.")

    df_valid["z_val_accuracy"] = zscore_safe(df_valid["val_accuracy"])
    df_valid["z_dsi"] = zscore_safe(df_valid["dsi"])
    df_valid["hybrid_score"] = df_valid["z_val_accuracy"] - df_valid["z_dsi"]

    best_row = df_valid.loc[df_valid["hybrid_score"].idxmax()]
    score_threshold = df_valid["hybrid_score"].median()
    hybrid_configs = int((df_valid["hybrid_score"] >= score_threshold).sum())

    return float(best_row["learning_rate"]), hybrid_configs

# ==========================================================
# BLOCK 22 — FINAL EPOCHS BY DATASET/ARCHITECTURE
# ==========================================================

def resolve_final_epochs(dataset_name: str, architecture_name: str) -> int:
    dataset_name = dataset_name.lower()
    architecture_name = architecture_name.lower()

    if dataset_name in ["mnist", "fashionmnist", "wine", "breastcancer", "digits"]:
        return CONFIG.final_epochs_small

    if dataset_name in ["cifar10", "cifar100"]:
        if architecture_name == "resnet_like":
            return CONFIG.final_epochs_cifar_resnet
        return CONFIG.final_epochs_cifar_cnn

    return CONFIG.final_epochs_small

# ==========================================================
# BLOCK 23 — FINAL TRAINING AND EVALUATION
# ==========================================================

def train_final_and_test(
    model_builder: Callable[[], nn.Module],
    train_full_loader: DataLoader,
    test_loader: DataLoader,
    lr: float,
    optimizer_name: str = "adam",
    epochs: int = CONFIG.final_epochs_small,
    scheduler_name: str = "none",
    run_seed: Optional[int] = None
) -> Dict[str, float]:
    out = run_training_trajectory(
        model_builder=model_builder,
        train_loader=train_full_loader,
        lr=float(lr),
        optimizer_name=optimizer_name,
        epochs=epochs,
        record_dynamics=False,
        scheduler_name=scheduler_name,
        run_seed=run_seed
    )

    if out["diverged"]:
        test_acc = np.nan
    else:
        test_acc = evaluate_accuracy(out["model"], test_loader)

    return {
        "test_accuracy": test_acc,
        "train_time_seconds": out["elapsed_seconds"],
        "batches_processed": out["batches_processed"],
        "diverged": out["diverged"]
    }

def train_and_evaluate_holdout(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    eval_loader: DataLoader,
    lr: float,
    optimizer_name: str = "adam",
    epochs: int = CONFIG.main_epochs,
    scheduler_name: str = "none",
    run_seed: Optional[int] = None
) -> Dict[str, float]:
    out = run_training_trajectory(
        model_builder=model_builder,
        train_loader=train_loader,
        lr=float(lr),
        optimizer_name=optimizer_name,
        epochs=epochs,
        record_dynamics=False,
        scheduler_name=scheduler_name,
        run_seed=run_seed
    )

    if out["diverged"]:
        eval_acc = np.nan
    else:
        eval_acc = evaluate_accuracy(out["model"], eval_loader)

    return {
        "holdout_accuracy": eval_acc,
        "train_time_seconds": out["elapsed_seconds"],
        "batches_processed": out["batches_processed"],
        "diverged": out["diverged"]
    }

# ==========================================================
# BLOCK 24 — DYNAMIC RULE SENSITIVITY
# ==========================================================

def run_dynamic_selection_sensitivity(
    df_dynamic: pd.DataFrame,
    quantiles: Tuple[float, ...] = CONFIG.dynamic_quantiles
) -> pd.DataFrame:
    rows = []

    try:
        lr_min_dsi, cfg_min_dsi = select_lr_by_dynamic_pure(df_dynamic, rule_name="min_dsi")
        rows.append({
            "rule": "min_dsi",
            "quantile": np.nan,
            "selected_lr": lr_min_dsi,
            "configs_considered": cfg_min_dsi
        })
    except Exception:
        rows.append({
            "rule": "min_dsi",
            "quantile": np.nan,
            "selected_lr": np.nan,
            "configs_considered": np.nan
        })

    for q in quantiles:
        try:
            lr_q, cfg_q = select_lr_by_dynamic_pure(
                df_dynamic=df_dynamic,
                rule_name="quantile_plus_descent",
                quantile=q
            )
            rows.append({
                "rule": "quantile_plus_descent",
                "quantile": q,
                "selected_lr": lr_q,
                "configs_considered": cfg_q
            })
        except Exception:
            rows.append({
                "rule": "quantile_plus_descent",
                "quantile": q,
                "selected_lr": np.nan,
                "configs_considered": np.nan
            })

    try:
        lr_rank, cfg_rank = select_lr_by_dynamic_pure(df_dynamic, rule_name="rank_combined")
        rows.append({
            "rule": "rank_combined",
            "quantile": np.nan,
            "selected_lr": lr_rank,
            "configs_considered": cfg_rank
        })
    except Exception:
        rows.append({
            "rule": "rank_combined",
            "quantile": np.nan,
            "selected_lr": np.nan,
            "configs_considered": np.nan
        })

    return pd.DataFrame(rows)

def evaluate_dynamic_selection_sensitivity_downstream(
    dataset_name: str,
    architecture_name: str,
    model_builder: Callable[[], nn.Module],
    train_search_loader: DataLoader,
    val_loader: DataLoader,
    df_dynamic: pd.DataFrame,
    optimizer_name: str = "adam",
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    eval_epochs = resolve_final_epochs(dataset_name, architecture_name)
    sensitivity_df = run_dynamic_selection_sensitivity(df_dynamic)

    rows = []

    for _, row in sensitivity_df.iterrows():
        rule = row["rule"]
        quantile = row["quantile"]
        selected_lr = row["selected_lr"]
        configs_considered = row["configs_considered"]

        if pd.isna(selected_lr):
            rows.append({
                "rule": rule,
                "quantile": quantile,
                "selected_lr": np.nan,
                "configs_considered": configs_considered,
                "holdout_val_accuracy": np.nan,
                "holdout_train_time_seconds": np.nan,
                "holdout_train_batches": np.nan
            })
            continue

        eval_out = train_and_evaluate_holdout(
            model_builder=model_builder,
            train_loader=train_search_loader,
            eval_loader=val_loader,
            lr=float(selected_lr),
            optimizer_name=optimizer_name,
            epochs=eval_epochs,
            scheduler_name="none",
            run_seed=run_seed
        )

        rows.append({
            "rule": rule,
            "quantile": quantile,
            "selected_lr": selected_lr,
            "configs_considered": configs_considered,
            "holdout_val_accuracy": eval_out["holdout_accuracy"],
            "holdout_train_time_seconds": eval_out["train_time_seconds"],
            "holdout_train_batches": eval_out["batches_processed"]
        })

    return pd.DataFrame(rows)

# ==========================================================
# BLOCK 24.5 — ROBUSTNESS HELPERS
# ==========================================================

def resolve_dynamic_epochs_for_dsi(
    train_loader: DataLoader,
    min_points: int = CONFIG.min_series_points,
    max_batches: Optional[int] = CONFIG.dynamic_search_max_batches
) -> int:
    batches_per_epoch = len(train_loader)

    if batches_per_epoch <= 0:
        return 1

    effective_batches_per_epoch = batches_per_epoch
    if max_batches is not None:
        effective_batches_per_epoch = min(effective_batches_per_epoch, max_batches)

    return max(1, math.ceil(min_points / effective_batches_per_epoch))

def is_valid_lr_value(lr: Any) -> bool:
    try:
        return lr is not None and np.isfinite(float(lr))
    except Exception:
        return False

def safe_train_final_and_test(
    model_builder: Callable[[], nn.Module],
    train_full_loader: DataLoader,
    test_loader: DataLoader,
    lr: Any,
    optimizer_name: str = "adam",
    epochs: int = CONFIG.final_epochs_small,
    scheduler_name: str = "none",
    run_seed: Optional[int] = None
) -> Dict[str, float]:
    if not is_valid_lr_value(lr):
        return {
            "test_accuracy": np.nan,
            "train_time_seconds": np.nan,
            "batches_processed": np.nan,
            "diverged": np.nan
        }

    return train_final_and_test(
        model_builder=model_builder,
        train_full_loader=train_full_loader,
        test_loader=test_loader,
        lr=float(lr),
        optimizer_name=optimizer_name,
        epochs=epochs,
        scheduler_name=scheduler_name,
        run_seed=run_seed
    )

def get_method_metric(comparison: pd.DataFrame, method_name: str, metric_name: str) -> float:
    rows = comparison.loc[comparison["method"] == method_name, metric_name]
    if len(rows) == 0:
        return np.nan
    return rows.values[0]

# ==========================================================
# BLOCK 25 — METHOD COMPARISON EXPERIMENT
# ==========================================================

def run_method_comparison_experiment(
    dataset_name: str,
    architecture_name: str,
    model_builder: Callable[[], nn.Module],
    train_search_loader: DataLoader,
    val_loader: DataLoader,
    train_full_loader: DataLoader,
    test_loader: DataLoader,
    learning_rates: np.ndarray,
    optimizer_name: str = "adam",
    dynamic_rule_name: str = "quantile_plus_descent",
    dynamic_quantile: float = 0.33,
    dsi_params: Optional[Dict[str, float]] = None,
    run_seed: Optional[int] = None
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if dsi_params is None:
        dsi_params = {}

    dynamic_epochs = resolve_dynamic_epochs_for_dsi(
        train_loader=train_search_loader,
        min_points=CONFIG.min_series_points,
        max_batches=CONFIG.dynamic_search_max_batches
    )

    print(
        f"[{dataset_name} | {architecture_name}] "
        f"search_batches={len(train_search_loader)} | dynamic_epochs={dynamic_epochs}"
    )

    df_dynamic, dynamic_trajectories = run_lr_sweep(
        model_builder=model_builder,
        train_loader=train_search_loader,
        learning_rates=learning_rates,
        optimizer_name=optimizer_name,
        epochs=dynamic_epochs,
        scheduler_name="none",
        max_batches=CONFIG.dynamic_search_max_batches,
        dsi_params=dsi_params,
        run_seed=run_seed
    )

    df_val = run_validation_accuracy_sweep(
        model_builder=model_builder,
        train_loader=train_search_loader,
        val_loader=val_loader,
        learning_rates=learning_rates,
        optimizer_name=optimizer_name,
        epochs=CONFIG.exploratory_epochs,
        scheduler_name="none",
        run_seed=run_seed
    )

    df_merged = df_dynamic.merge(
        df_val[["learning_rate", "optimizer", "scheduler", "val_accuracy"]],
        on=["learning_rate", "optimizer", "scheduler"],
        how="left"
    )

    df_early = run_early_loss_sweep_baseline(
        model_builder=model_builder,
        train_loader=train_search_loader,
        learning_rates=learning_rates,
        optimizer_name=optimizer_name,
        run_seed=run_seed
    )

    if CONFIG.run_selection_sensitivity:
        df_sensitivity = run_dynamic_selection_sensitivity(df_dynamic)
        df_sensitivity_downstream = evaluate_dynamic_selection_sensitivity_downstream(
            dataset_name=dataset_name,
            architecture_name=architecture_name,
            model_builder=model_builder,
            train_search_loader=train_search_loader,
            val_loader=val_loader,
            df_dynamic=df_dynamic,
            optimizer_name=optimizer_name,
            run_seed=run_seed
        )
    else:
        df_sensitivity = pd.DataFrame()
        df_sensitivity_downstream = pd.DataFrame()

    try:
        grid_lr = select_lr_by_grid_search(df_val)
    except Exception as e:
        print(f"[WARN] Grid Search failed in {dataset_name}/{architecture_name}: {e}")
        grid_lr = np.nan

    try:
        dynamic_lr, dynamic_configs = select_lr_by_dynamic_pure(
            df_dynamic=df_dynamic,
            rule_name=dynamic_rule_name,
            quantile=dynamic_quantile
        )
    except Exception as e:
        print(f"[WARN] Dynamic Criterion failed in {dataset_name}/{architecture_name}: {e}")
        dynamic_lr, dynamic_configs = np.nan, np.nan

    try:
        early_lr = select_lr_by_early_loss_sweep(df_early)
    except Exception as e:
        print(f"[WARN] Early-loss Sweep failed in {dataset_name}/{architecture_name}: {e}")
        early_lr = np.nan

    try:
        random_lr, df_random = run_random_search_validation(
            model_builder=model_builder,
            train_loader=train_search_loader,
            val_loader=val_loader,
            learning_rates=learning_rates,
            optimizer_name=optimizer_name,
            epochs=CONFIG.exploratory_epochs,
            n_samples=CONFIG.random_search_samples,
            sampling_seed=run_seed if run_seed is not None else CONFIG.seed,
            run_seed=run_seed
        )
    except Exception as e:
        print(f"[WARN] Random Search failed in {dataset_name}/{architecture_name}: {e}")
        random_lr = np.nan
        df_random = pd.DataFrame(columns=[
            "learning_rate", "optimizer", "scheduler", "val_accuracy",
            "batches_processed", "elapsed_seconds", "diverged"
        ])

    if CONFIG.run_hybrid_method:
        try:
            hybrid_lr, hybrid_configs = select_lr_by_hybrid_score(df_merged)
        except Exception as e:
            print(f"[WARN] Hybrid Criterion failed in {dataset_name}/{architecture_name}: {e}")
            hybrid_lr, hybrid_configs = np.nan, np.nan
    else:
        hybrid_lr, hybrid_configs = np.nan, np.nan

    grid_search_time = df_val["elapsed_seconds"].sum() if len(df_val) > 0 else np.nan
    dynamic_search_time = df_dynamic["elapsed_seconds"].sum() if len(df_dynamic) > 0 else np.nan
    early_search_time = df_early["elapsed_seconds"].sum() if len(df_early) > 0 else np.nan
    random_search_time = df_random["elapsed_seconds"].sum() if len(df_random) > 0 else np.nan

    grid_search_batches = df_val["batches_processed"].sum() if len(df_val) > 0 else np.nan
    dynamic_search_batches = df_dynamic["batches_processed"].sum() if len(df_dynamic) > 0 else np.nan
    early_search_batches = df_early["batches_processed"].sum() if len(df_early) > 0 else np.nan
    random_search_batches = df_random["batches_processed"].sum() if len(df_random) > 0 else np.nan

    if CONFIG.run_hybrid_method:
        hybrid_search_time = (
            (df_dynamic["elapsed_seconds"].sum() if len(df_dynamic) > 0 else 0.0) +
            (df_val["elapsed_seconds"].sum() if len(df_val) > 0 else 0.0)
        )
        hybrid_search_batches = (
            (df_dynamic["batches_processed"].sum() if len(df_dynamic) > 0 else 0.0) +
            (df_val["batches_processed"].sum() if len(df_val) > 0 else 0.0)
        )
    else:
        hybrid_search_time = np.nan
        hybrid_search_batches = np.nan

    final_epochs = resolve_final_epochs(dataset_name, architecture_name)

    results_rows = []

    grid_eval = safe_train_final_and_test(
        model_builder=model_builder,
        train_full_loader=train_full_loader,
        test_loader=test_loader,
        lr=grid_lr,
        optimizer_name=optimizer_name,
        epochs=final_epochs,
        scheduler_name="none",
        run_seed=run_seed
    )
    results_rows.append({
        "dataset": dataset_name,
        "architecture": architecture_name,
        "method": "Grid Search",
        "learning_rate": grid_lr,
        "scheduler": "none",
        "test_accuracy": grid_eval["test_accuracy"],
        "search_time_seconds": grid_search_time,
        "search_batches": grid_search_batches,
        "final_train_time_seconds": grid_eval["train_time_seconds"],
        "final_train_batches": grid_eval["batches_processed"],
        "final_diverged": grid_eval["diverged"]
    })

    dyn_eval = safe_train_final_and_test(
        model_builder=model_builder,
        train_full_loader=train_full_loader,
        test_loader=test_loader,
        lr=dynamic_lr,
        optimizer_name=optimizer_name,
        epochs=final_epochs,
        scheduler_name="none",
        run_seed=run_seed
    )
    results_rows.append({
        "dataset": dataset_name,
        "architecture": architecture_name,
        "method": "Dynamic Criterion",
        "learning_rate": dynamic_lr,
        "scheduler": "none",
        "test_accuracy": dyn_eval["test_accuracy"],
        "search_time_seconds": dynamic_search_time,
        "search_batches": dynamic_search_batches,
        "final_train_time_seconds": dyn_eval["train_time_seconds"],
        "final_train_batches": dyn_eval["batches_processed"],
        "final_diverged": dyn_eval["diverged"]
    })

    if CONFIG.run_hybrid_method:
        hyb_eval = safe_train_final_and_test(
            model_builder=model_builder,
            train_full_loader=train_full_loader,
            test_loader=test_loader,
            lr=hybrid_lr,
            optimizer_name=optimizer_name,
            epochs=final_epochs,
            scheduler_name="none",
            run_seed=run_seed
        )
        results_rows.append({
            "dataset": dataset_name,
            "architecture": architecture_name,
            "method": "Hybrid Criterion",
            "learning_rate": hybrid_lr,
            "scheduler": "none",
            "test_accuracy": hyb_eval["test_accuracy"],
            "search_time_seconds": hybrid_search_time,
            "search_batches": hybrid_search_batches,
            "final_train_time_seconds": hyb_eval["train_time_seconds"],
            "final_train_batches": hyb_eval["batches_processed"],
            "final_diverged": hyb_eval["diverged"]
        })

    early_eval = safe_train_final_and_test(
        model_builder=model_builder,
        train_full_loader=train_full_loader,
        test_loader=test_loader,
        lr=early_lr,
        optimizer_name=optimizer_name,
        epochs=final_epochs,
        scheduler_name="none",
        run_seed=run_seed
    )
    results_rows.append({
        "dataset": dataset_name,
        "architecture": architecture_name,
        "method": "Early-loss Sweep",
        "learning_rate": early_lr,
        "scheduler": "none",
        "test_accuracy": early_eval["test_accuracy"],
        "search_time_seconds": early_search_time,
        "search_batches": early_search_batches,
        "final_train_time_seconds": early_eval["train_time_seconds"],
        "final_train_batches": early_eval["batches_processed"],
        "final_diverged": early_eval["diverged"]
    })

    random_eval = safe_train_final_and_test(
        model_builder=model_builder,
        train_full_loader=train_full_loader,
        test_loader=test_loader,
        lr=random_lr,
        optimizer_name=optimizer_name,
        epochs=final_epochs,
        scheduler_name="none",
        run_seed=run_seed
    )
    results_rows.append({
        "dataset": dataset_name,
        "architecture": architecture_name,
        "method": "Random Search",
        "learning_rate": random_lr,
        "scheduler": "none",
        "test_accuracy": random_eval["test_accuracy"],
        "search_time_seconds": random_search_time,
        "search_batches": random_search_batches,
        "final_train_time_seconds": random_eval["train_time_seconds"],
        "final_train_batches": random_eval["batches_processed"],
        "final_diverged": random_eval["diverged"]
    })

    if CONFIG.run_scheduler_baselines:
        step_eval = safe_train_final_and_test(
            model_builder=model_builder,
            train_full_loader=train_full_loader,
            test_loader=test_loader,
            lr=grid_lr,
            optimizer_name=optimizer_name,
            epochs=final_epochs,
            scheduler_name="steplr",
            run_seed=run_seed
        )
        results_rows.append({
            "dataset": dataset_name,
            "architecture": architecture_name,
            "method": "StepLR",
            "learning_rate": grid_lr,
            "scheduler": "StepLR",
            "test_accuracy": step_eval["test_accuracy"],
            "search_time_seconds": grid_search_time,
            "search_batches": grid_search_batches,
            "final_train_time_seconds": step_eval["train_time_seconds"],
            "final_train_batches": step_eval["batches_processed"],
            "final_diverged": step_eval["diverged"]
        })

        cosine_eval = safe_train_final_and_test(
            model_builder=model_builder,
            train_full_loader=train_full_loader,
            test_loader=test_loader,
            lr=grid_lr,
            optimizer_name=optimizer_name,
            epochs=final_epochs,
            scheduler_name="cosineannealinglr",
            run_seed=run_seed
        )
        results_rows.append({
            "dataset": dataset_name,
            "architecture": architecture_name,
            "method": "CosineAnnealingLR",
            "learning_rate": grid_lr,
            "scheduler": "CosineAnnealingLR",
            "test_accuracy": cosine_eval["test_accuracy"],
            "search_time_seconds": grid_search_time,
            "search_batches": grid_search_batches,
            "final_train_time_seconds": cosine_eval["train_time_seconds"],
            "final_train_batches": cosine_eval["batches_processed"],
            "final_diverged": cosine_eval["diverged"]
        })

    comparison = pd.DataFrame(results_rows)

    comparison["accuracy_per_search_second"] = comparison.apply(
        lambda row: safe_divide(row["test_accuracy"], row["search_time_seconds"]),
        axis=1
    )
    comparison["accuracy_per_100_batches"] = comparison.apply(
        lambda row: safe_divide(row["test_accuracy"], row["search_batches"] / 100.0),
        axis=1
    )

    summary = pd.DataFrame([{
        "dataset": dataset_name,
        "architecture": architecture_name,
        "dynamic_rule_name": dynamic_rule_name,
        "dynamic_quantile": dynamic_quantile,
        "grid_lr": grid_lr,
        "dynamic_lr": dynamic_lr,
        "early_lr": early_lr,
        "random_lr": random_lr,
        "hybrid_lr": hybrid_lr if CONFIG.run_hybrid_method else np.nan,
        "grid_test_accuracy": get_method_metric(comparison, "Grid Search", "test_accuracy"),
        "dynamic_test_accuracy": get_method_metric(comparison, "Dynamic Criterion", "test_accuracy"),
        "early_test_accuracy": get_method_metric(comparison, "Early-loss Sweep", "test_accuracy"),
        "random_test_accuracy": get_method_metric(comparison, "Random Search", "test_accuracy"),
        "grid_search_time_seconds": get_method_metric(comparison, "Grid Search", "search_time_seconds"),
        "dynamic_search_time_seconds": get_method_metric(comparison, "Dynamic Criterion", "search_time_seconds")
    }])

    return df_merged, comparison, summary, df_sensitivity, df_sensitivity_downstream

# ==========================================================
# BLOCK 26 — DSI/PERFORMANCE CORRELATION
# ==========================================================

def compute_dsi_performance_correlation(df_search: pd.DataFrame) -> pd.DataFrame:
    df_valid = df_search.dropna(subset=["dsi", "val_accuracy"]).copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]

    if len(df_valid) < 3:
        return pd.DataFrame([{
            "pearson_corr_dsi_valacc": np.nan,
            "spearman_corr_dsi_valacc": np.nan
        }])

    pearson_corr = df_valid["dsi"].corr(df_valid["val_accuracy"], method="pearson")
    spearman_corr = df_valid["dsi"].corr(df_valid["val_accuracy"], method="spearman")

    return pd.DataFrame([{
        "pearson_corr_dsi_valacc": pearson_corr,
        "spearman_corr_dsi_valacc": spearman_corr
    }])

# ==========================================================
# BLOCK 27 — MULTI-ARCHITECTURE IMAGE EXPERIMENT
# ==========================================================

def run_image_optimizer_architecture_experiment(
    dataset_name: str,
    architecture_name: str,
    learning_rates: np.ndarray,
    optimizers: List[str] = ("sgd", "adam", "rmsprop"),
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    dataset_name = dataset_name.lower()
    architecture_name = architecture_name.lower()
    local_seed = run_seed if run_seed is not None else CONFIG.seed

    if dataset_name == "cifar10":
        train_loader, val_loader, test_loader, train_full_loader = get_cifar10_loaders(seed=local_seed)
        if architecture_name == "mlp":
            model_builder = lambda: build_cifar_mlp(output_dim=10)
        elif architecture_name == "cnn":
            model_builder = build_cifar10_cnn
        elif architecture_name == "resnet_like":
            model_builder = build_cifar10_resnet_like
        else:
            raise ValueError("Unsupported architecture for CIFAR-10")

    elif dataset_name == "cifar100":
        train_loader, val_loader, test_loader, train_full_loader = get_cifar100_loaders(seed=local_seed)
        if architecture_name == "mlp":
            model_builder = lambda: build_cifar_mlp(output_dim=100)
        elif architecture_name == "cnn":
            model_builder = build_cifar100_cnn
        elif architecture_name == "resnet_like":
            model_builder = build_cifar100_resnet_like
        else:
            raise ValueError("Unsupported architecture for CIFAR-100")
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

    rows = []

    for optimizer_name in optimizers:
        for lr in learning_rates:
            out = run_training_trajectory(
                model_builder=model_builder,
                train_loader=train_loader,
                lr=float(lr),
                optimizer_name=optimizer_name,
                epochs=CONFIG.main_epochs,
                record_dynamics=True,
                scheduler_name="none",
                run_seed=local_seed
            )

            model = out["model"]
            losses = out["loss_series"]

            dsi_dict = dynamic_stability_index(losses, out["has_non_finite"])
            val_acc = np.nan if out["diverged"] else evaluate_accuracy(model, val_loader)

            rows.append({
                "dataset": dataset_name,
                "architecture": architecture_name,
                "optimizer": optimizer_name,
                "learning_rate": float(lr),
                "dsi": dsi_dict["dsi"],
                "volatility": dsi_dict["volatility"],
                "oscillation": dsi_dict["oscillation"],
                "descent": dsi_dict["descent"],
                "instability_penalty": dsi_dict["instability_penalty"],
                "val_accuracy": val_acc,
                "has_non_finite": out["has_non_finite"],
                "diverged": out["diverged"],
                "elapsed_seconds": out["elapsed_seconds"],
                "batches_processed": out["batches_processed"]
            })

    return pd.DataFrame(rows)

# ==========================================================
# BLOCK 28 — MULTI-DATASET EXPERIMENT
# ==========================================================

def run_multidataset_experiment(run_seed: int = CONFIG.seed) -> pd.DataFrame:
    results = []

    transform = transforms.Compose([transforms.ToTensor()])

    mnist_train = torchvision.datasets.MNIST(CONFIG.data_dir, train=True, download=True, transform=transform)
    mnist_test = torchvision.datasets.MNIST(CONFIG.data_dir, train=False, download=True, transform=transform)

    fashion_train = torchvision.datasets.FashionMNIST(CONFIG.data_dir, train=True, download=True, transform=transform)
    fashion_test = torchvision.datasets.FashionMNIST(CONFIG.data_dir, train=False, download=True, transform=transform)

    vision_sets = {
        "MNIST": (mnist_train, mnist_test),
        "FashionMNIST": (fashion_train, fashion_test)
    }

    for dataset_name, (train_ds, test_ds) in vision_sets.items():
        print(f"[MULTI-DATASET] Running {dataset_name}")
        train_subset = random_subset(train_ds, CONFIG.multidataset_vision_subset, run_seed)

        n_total = len(train_subset)
        n_train = int(0.8 * n_total)
        n_val = n_total - n_train
        generator = torch.Generator().manual_seed(run_seed)

        train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

        train_search_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=run_seed)
        val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=run_seed)
        train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=run_seed)
        test_loader = make_loader(test_ds, CONFIG.batch_size_eval, False, seed=run_seed)

        _, comparison, summary, _, _ = run_method_comparison_experiment(
            dataset_name=dataset_name,
            architecture_name="mlp",
            model_builder=build_simple_image_mlp,
            train_search_loader=train_search_loader,
            val_loader=val_loader,
            train_full_loader=train_full_loader,
            test_loader=test_loader,
            learning_rates=np.logspace(-4, 0, 10),
            optimizer_name="adam",
            dynamic_rule_name="quantile_plus_descent",
            dynamic_quantile=0.33,
            run_seed=run_seed
        )

        results.append(summary.iloc[0].to_dict())

    tabular_names = ["wine", "breastcancer", "digits"]

    for dataset_name in tabular_names:
        print(f"[MULTI-DATASET] Running {dataset_name}")
        train_search_loader, val_loader, test_loader, train_full_loader, metadata = get_tabular_loaders(dataset_name, seed=run_seed)

        model_builder = lambda metadata=metadata: TabularMLP(
            input_dim=metadata["input_dim"],
            output_dim=metadata["n_classes"]
        )

        _, comparison, summary, _, _ = run_method_comparison_experiment(
            dataset_name=dataset_name,
            architecture_name="tabular_mlp",
            model_builder=model_builder,
            train_search_loader=train_search_loader,
            val_loader=val_loader,
            train_full_loader=train_full_loader,
            test_loader=test_loader,
            learning_rates=np.logspace(-4, 0, 10),
            optimizer_name="adam",
            dynamic_rule_name="quantile_plus_descent",
            dynamic_quantile=0.33,
            run_seed=run_seed
        )

        results.append(summary.iloc[0].to_dict())

    return pd.DataFrame(results)

# ==========================================================
# BLOCK 29 — MULTI-SEED VALIDATION
# ==========================================================

def run_multiseed_validation() -> pd.DataFrame:
    all_runs = []

    transform = transforms.Compose([transforms.ToTensor()])

    vision_sets = {
        "MNIST": (
            torchvision.datasets.MNIST(CONFIG.data_dir, train=True, download=True, transform=transform),
            torchvision.datasets.MNIST(CONFIG.data_dir, train=False, download=True, transform=transform)
        ),
        "FashionMNIST": (
            torchvision.datasets.FashionMNIST(CONFIG.data_dir, train=True, download=True, transform=transform),
            torchvision.datasets.FashionMNIST(CONFIG.data_dir, train=False, download=True, transform=transform)
        )
    }

    tabular_sets = ["wine", "breastcancer", "digits"]

    for seed in CONFIG.seeds_multirun:
        print(f"[MULTI-SEED] Seed {seed}")
        set_global_seed(seed)

        for dataset_name, (train_ds, test_ds) in vision_sets.items():
            print(f"[MULTI-SEED] Seed={seed} | Dataset={dataset_name}")
            train_subset = random_subset(train_ds, CONFIG.multidataset_vision_subset, seed)

            n_total = len(train_subset)
            n_train = int(0.8 * n_total)
            n_val = n_total - n_train
            generator = torch.Generator().manual_seed(seed)

            train_inner, val_inner = random_split(train_subset, [n_train, n_val], generator=generator)

            train_search_loader = make_loader(train_inner, CONFIG.batch_size_train, True, seed=seed)
            val_loader = make_loader(val_inner, CONFIG.batch_size_eval, False, seed=seed)
            train_full_loader = make_loader(train_subset, CONFIG.batch_size_train, True, seed=seed)
            test_loader = make_loader(test_ds, CONFIG.batch_size_eval, False, seed=seed)

            _, _, summary, _, _ = run_method_comparison_experiment(
                dataset_name=dataset_name,
                architecture_name="mlp",
                model_builder=build_simple_image_mlp,
                train_search_loader=train_search_loader,
                val_loader=val_loader,
                train_full_loader=train_full_loader,
                test_loader=test_loader,
                learning_rates=np.logspace(-4, 0, 10),
                optimizer_name="adam",
                dynamic_rule_name="quantile_plus_descent",
                dynamic_quantile=0.33,
                run_seed=seed
            )

            row = summary.iloc[0].to_dict()
            row["seed"] = seed
            all_runs.append(row)

        for dataset_name in tabular_sets:
            print(f"[MULTI-SEED] Seed={seed} | Dataset={dataset_name}")
            train_search_loader, val_loader, test_loader, train_full_loader, metadata = get_tabular_loaders(dataset_name, seed=seed)

            model_builder = lambda metadata=metadata: TabularMLP(
                input_dim=metadata["input_dim"],
                output_dim=metadata["n_classes"]
            )

            _, _, summary, _, _ = run_method_comparison_experiment(
                dataset_name=dataset_name,
                architecture_name="tabular_mlp",
                model_builder=model_builder,
                train_search_loader=train_search_loader,
                val_loader=val_loader,
                train_full_loader=train_full_loader,
                test_loader=test_loader,
                learning_rates=np.logspace(-4, 0, 10),
                optimizer_name="adam",
                dynamic_rule_name="quantile_plus_descent",
                dynamic_quantile=0.33,
                run_seed=seed
            )

            row = summary.iloc[0].to_dict()
            row["seed"] = seed
            all_runs.append(row)

    return pd.DataFrame(all_runs)

def summarize_multiseed(df_multiseed: pd.DataFrame) -> pd.DataFrame:
    summary = df_multiseed.groupby(["dataset", "architecture"]).agg(
        grid_test_accuracy_mean=("grid_test_accuracy", "mean"),
        grid_test_accuracy_std=("grid_test_accuracy", "std"),
        dynamic_test_accuracy_mean=("dynamic_test_accuracy", "mean"),
        dynamic_test_accuracy_std=("dynamic_test_accuracy", "std"),
        random_test_accuracy_mean=("random_test_accuracy", "mean"),
        random_test_accuracy_std=("random_test_accuracy", "std"),
        grid_search_time_mean=("grid_search_time_seconds", "mean"),
        grid_search_time_std=("grid_search_time_seconds", "std"),
        dynamic_search_time_mean=("dynamic_search_time_seconds", "mean"),
        dynamic_search_time_std=("dynamic_search_time_seconds", "std")
    ).reset_index()

    n = len(CONFIG.seeds_multirun)
    summary["dynamic_acc_ci95"] = 1.96 * (summary["dynamic_test_accuracy_std"] / np.sqrt(n))
    summary["grid_acc_ci95"] = 1.96 * (summary["grid_test_accuracy_std"] / np.sqrt(n))

    return summary.round(4)

def paired_statistical_tests(df_multiseed: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for (dataset, architecture), group in df_multiseed.groupby(["dataset", "architecture"]):
        x = group["grid_test_accuracy"].to_numpy(dtype=float)
        y = group["dynamic_test_accuracy"].to_numpy(dtype=float)

        mask = np.isfinite(x) & np.isfinite(y)
        x = x[mask]
        y = y[mask]

        if len(x) < 2:
            rows.append({
                "dataset": dataset,
                "architecture": architecture,
                "n_pairs": len(x),
                "ttest_pvalue": np.nan,
                "wilcoxon_pvalue": np.nan,
                "cohen_d_paired": np.nan
            })
            continue

        try:
            t_p = ttest_rel(x, y, nan_policy="omit").pvalue
        except Exception:
            t_p = np.nan

        try:
            w_p = wilcoxon(x, y).pvalue
        except Exception:
            w_p = np.nan

        d_eff = cohen_d_paired(x, y)

        rows.append({
            "dataset": dataset,
            "architecture": architecture,
            "n_pairs": len(x),
            "ttest_pvalue": t_p,
            "wilcoxon_pvalue": w_p,
            "cohen_d_paired": d_eff
        })

    return pd.DataFrame(rows)

# ==========================================================
# BLOCK 30 — FIGURES
# ==========================================================

def plot_dsi_vs_lr(df_summary: pd.DataFrame, filename: str):
    df_plot = df_summary.dropna(subset=["dsi"]).copy()
    df_plot = df_plot[np.isfinite(df_plot["dsi"])]
    if len(df_plot) == 0:
        return

    fig = px.line(
        df_plot,
        x="learning_rate",
        y="dsi",
        markers=True,
        title="Dynamic Stability Index versus Learning Rate"
    )

    fig.update_xaxes(type="log")
    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Learning rate",
        yaxis_title="Dynamic Stability Index (DSI)"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_loss_dynamics(trajectories: Dict[float, Dict[str, Any]], selected_lrs: List[float], filename: str):
    rows = []

    for lr in selected_lrs:
        if lr not in trajectories:
            continue

        series = trajectories[lr]["loss_series"][:200]
        if len(series) == 0:
            continue

        for i, loss_value in enumerate(series):
            rows.append({
                "iteration": i,
                "loss": loss_value,
                "learning_rate": f"{lr:.4g}"
            })

    if len(rows) == 0:
        return

    df_plot = pd.DataFrame(rows)

    fig = px.line(
        df_plot,
        x="iteration",
        y="loss",
        color="learning_rate",
        title="Training-loss dynamics for selected learning rates"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Iteration",
        yaxis_title="Loss"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_phase_space(trajectories: Dict[float, Dict[str, Any]], lr_example: float, filename: str):
    if lr_example not in trajectories:
        return

    series = trajectories[lr_example]["loss_series"]
    if len(series) < 3:
        return

    df_phase = pd.DataFrame({
        "loss_t": series[:-1],
        "loss_t1": series[1:]
    })

    fig = px.scatter(
        df_phase,
        x="loss_t",
        y="loss_t1",
        title=f"Training phase-space view (lr={lr_example:.4g})"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="loss(t)",
        yaxis_title="loss(t+1)"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_validation_accuracy_vs_lr(df_acc: pd.DataFrame, filename: str):
    df_plot = df_acc.dropna(subset=["val_accuracy"]).copy()
    if len(df_plot) == 0:
        return

    best_row = df_plot.loc[df_plot["val_accuracy"].idxmax()]

    fig = px.line(
        df_plot,
        x="learning_rate",
        y="val_accuracy",
        markers=True,
        title="Validation accuracy versus learning rate"
    )

    fig.update_xaxes(type="log")
    fig.add_vline(
        x=best_row["learning_rate"],
        line_dash="dash",
        line_color="green",
        annotation_text="Best validation LR"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Learning rate",
        yaxis_title="Validation accuracy"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_3d_training_trajectory(trajectories: Dict[float, Dict[str, Any]], lr_example: float, filename: str):
    if lr_example not in trajectories:
        return

    losses = np.array(trajectories[lr_example]["loss_series"])
    weights = np.array(trajectories[lr_example]["weight_norm_series"])
    grads = np.array(trajectories[lr_example]["grad_norm_series"])

    if len(losses) < 3 or len(weights) < 3 or len(grads) < 3:
        return

    step = max(1, len(losses) // 1500)

    losses = losses[::step]
    weights = weights[::step]
    grads = grads[::step]
    time_index = np.arange(len(losses))

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=losses,
            y=weights,
            z=grads,
            mode="lines+markers",
            marker=dict(size=2, color=time_index, colorscale="Turbo", opacity=0.8),
            line=dict(width=2),
            name="Trajectory"
        )
    )

    fig.update_layout(
        title="Training trajectory in state space",
        scene=dict(
            xaxis_title="Loss",
            yaxis_title="Global weight norm",
            zaxis_title="Global gradient norm"
        ),
        template="plotly_white",
        title_x=0.5
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_dynamic_selection_scatter(df_search: pd.DataFrame, dynamic_lr: float, filename: str):
    df_plot = df_search.dropna(subset=["dsi", "val_accuracy"]).copy()
    df_plot = df_plot[np.isfinite(df_plot["dsi"])]
    if len(df_plot) == 0:
        return

    dsi_vals = df_plot.loc[df_plot["learning_rate"] == dynamic_lr, "dsi"].values
    if len(dsi_vals) == 0:
        return

    dynamic_dsi = dsi_vals[0]

    fig = px.scatter(
        df_plot,
        x="dsi",
        y="val_accuracy",
        title="DSI versus validation accuracy"
    )

    fig.add_vline(
        x=dynamic_dsi,
        line_dash="dash",
        line_color="black"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Dynamic Stability Index",
        yaxis_title="Validation accuracy"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

def plot_dsi_vs_accuracy(df: pd.DataFrame, filename: str):
    df_plot = df.dropna(subset=["dsi", "val_accuracy"]).copy()
    df_plot = df_plot[np.isfinite(df_plot["dsi"])]
    if len(df_plot) == 0:
        return

    fig = px.scatter(
        df_plot,
        x="dsi",
        y="val_accuracy",
        color="optimizer",
        title="DSI versus predictive performance"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Dynamic Stability Index",
        yaxis_title="Validation accuracy"
    )

    fig.write_html(os.path.join(CONFIG.output_dir, filename))
    fig.show()

# ==========================================================
# BLOCK 31 — MAIN PIPELINE
# ==========================================================

print("\n=== MAIN EXPERIMENT: MNIST (MLP) ===\n")

mnist_train_loader, mnist_val_loader, mnist_test_loader, mnist_train_full_loader = get_mnist_loaders(seed=CONFIG.seed)

df_mnist_dynamics, mnist_trajectories = run_lr_sweep(
    model_builder=build_mnist_mlp,
    train_loader=mnist_train_loader,
    learning_rates=LEARNING_RATES,
    optimizer_name="sgd",
    epochs=3,
    scheduler_name="none",
    run_seed=CONFIG.seed
)

df_mnist_val = run_validation_accuracy_sweep(
    model_builder=build_mnist_mlp,
    train_loader=mnist_train_loader,
    val_loader=mnist_val_loader,
    learning_rates=LEARNING_RATES,
    optimizer_name="sgd",
    epochs=3,
    scheduler_name="none",
    run_seed=CONFIG.seed
)

df_mnist_main = df_mnist_dynamics.merge(
    df_mnist_val[["learning_rate", "optimizer", "scheduler", "val_accuracy"]],
    on=["learning_rate", "optimizer", "scheduler"],
    how="left"
)

display(df_mnist_main)

save_table(df_mnist_main, "mnist_main_results", "MNIST main results")
save_pickle(mnist_trajectories, "mnist_trajectories.pkl")

plot_dsi_vs_lr(df_mnist_main, "mnist_dsi_vs_learning_rate.html")
selected_lrs = list(df_mnist_main["learning_rate"].values[:4])
plot_loss_dynamics(mnist_trajectories, selected_lrs, "mnist_loss_dynamics_selected_lrs.html")
lr_example = float(df_mnist_main["learning_rate"].values[min(3, len(df_mnist_main)-1)])
plot_phase_space(mnist_trajectories, lr_example, "mnist_phase_space_example.html")
plot_validation_accuracy_vs_lr(df_mnist_val, "mnist_validation_accuracy_vs_learning_rate.html")
lr_3d = float(df_mnist_main["learning_rate"].values[min(5, len(df_mnist_main)-1)])
plot_3d_training_trajectory(mnist_trajectories, lr_3d, "mnist_training_trajectory_3d.html")

if CONFIG.run_dsi_weight_sensitivity:
    df_mnist_dsi_weight_sensitivity = run_dsi_weight_sensitivity(
        trajectories=mnist_trajectories,
        rule_name="quantile_plus_descent",
        quantile=0.33
    )
    display(df_mnist_dsi_weight_sensitivity.head())
    save_table(
        df_mnist_dsi_weight_sensitivity,
        "mnist_dsi_weight_sensitivity",
        "MNIST DSI weight sensitivity"
    )
else:
    df_mnist_dsi_weight_sensitivity = pd.DataFrame()

print("\n=== METHOD COMPARISON: FASHIONMNIST ===\n")

fashion_train_loader, fashion_val_loader, fashion_test_loader, fashion_train_full_loader = get_fashion_mnist_loaders(seed=CONFIG.seed)

df_fashion_search, df_fashion_comparison, df_fashion_summary, df_fashion_sensitivity, df_fashion_sensitivity_downstream = run_method_comparison_experiment(
    dataset_name="FashionMNIST",
    architecture_name="cnn",
    model_builder=build_fashion_cnn,
    train_search_loader=fashion_train_loader,
    val_loader=fashion_val_loader,
    train_full_loader=fashion_train_full_loader,
    test_loader=fashion_test_loader,
    learning_rates=np.logspace(-4, 0, 12),
    optimizer_name="adam",
    dynamic_rule_name="quantile_plus_descent",
    dynamic_quantile=0.33,
    run_seed=CONFIG.seed
)

display(df_fashion_search.head())
display(df_fashion_comparison)
display(df_fashion_summary)
display(df_fashion_sensitivity)
display(df_fashion_sensitivity_downstream)

save_table(df_fashion_search, "fashion_search_results", "FashionMNIST search results")
save_table(df_fashion_comparison, "fashion_method_comparison", "FashionMNIST method comparison")
save_table(df_fashion_summary, "fashion_summary", "FashionMNIST summary")
save_table(df_fashion_sensitivity, "fashion_selection_sensitivity", "FashionMNIST selection sensitivity")
save_table(df_fashion_sensitivity_downstream, "fashion_selection_sensitivity_downstream", "FashionMNIST downstream sensitivity")

dynamic_lr_fashion = float(df_fashion_summary["dynamic_lr"].values[0]) if len(df_fashion_summary) > 0 else np.nan
if np.isfinite(dynamic_lr_fashion):
    plot_dynamic_selection_scatter(
        df_fashion_search,
        dynamic_lr=dynamic_lr_fashion,
        filename="fashion_dsi_vs_validation_accuracy.html"
    )

df_fashion_corr = compute_dsi_performance_correlation(df_fashion_search)
display(df_fashion_corr)
save_table(df_fashion_corr, "fashion_dsi_validation_correlation", "FashionMNIST DSI-validation correlation")

print("\n=== CIFAR-10 EXPERIMENT (CNN) ===\n")

df_cifar10_cnn = run_image_optimizer_architecture_experiment(
    dataset_name="cifar10",
    architecture_name="cnn",
    learning_rates=LEARNING_RATES,
    optimizers=["sgd", "adam", "rmsprop"],
    run_seed=CONFIG.seed
)

display(df_cifar10_cnn.head())
save_table(df_cifar10_cnn, "cifar10_cnn_results", "CIFAR-10 CNN results")
plot_dsi_vs_accuracy(df_cifar10_cnn, "cifar10_cnn_dsi_vs_accuracy.html")

print("\n=== CIFAR-10 EXPERIMENT (RESNET-LIKE) ===\n")

df_cifar10_resnet = run_image_optimizer_architecture_experiment(
    dataset_name="cifar10",
    architecture_name="resnet_like",
    learning_rates=LEARNING_RATES,
    optimizers=["sgd", "adam", "rmsprop"],
    run_seed=CONFIG.seed
)

display(df_cifar10_resnet.head())
save_table(df_cifar10_resnet, "cifar10_resnet_like_results", "CIFAR-10 ResNet-like results")
plot_dsi_vs_accuracy(df_cifar10_resnet, "cifar10_resnet_like_dsi_vs_accuracy.html")

if CONFIG.run_cifar100:
    print("\n=== CIFAR-100 EXPERIMENT (CNN) ===\n")

    df_cifar100_cnn = run_image_optimizer_architecture_experiment(
        dataset_name="cifar100",
        architecture_name="cnn",
        learning_rates=LEARNING_RATES,
        optimizers=["sgd", "adam", "rmsprop"],
        run_seed=CONFIG.seed
    )

    display(df_cifar100_cnn.head())
    save_table(df_cifar100_cnn, "cifar100_cnn_results", "CIFAR-100 CNN results")
    plot_dsi_vs_accuracy(df_cifar100_cnn, "cifar100_cnn_dsi_vs_accuracy.html")
else:
    df_cifar100_cnn = pd.DataFrame()

print("\n=== MULTI-DATASET EXPERIMENT ===\n")

df_multidataset = run_multidataset_experiment(run_seed=CONFIG.seed)
display(df_multidataset)
save_table(df_multidataset, "multidataset_summary", "Multi-dataset summary")

if CONFIG.run_multiseed:
    print("\n=== MULTI-SEED VALIDATION ===\n")

    df_multiseed = run_multiseed_validation()
    df_multiseed.to_csv(os.path.join(CONFIG.output_dir, "multiseed_all_runs.csv"), index=False)

    df_multiseed_summary = summarize_multiseed(df_multiseed)
    display(df_multiseed_summary)

    df_multiseed_tests = paired_statistical_tests(df_multiseed)
    display(df_multiseed_tests)

    save_table(df_multiseed_summary, "multiseed_summary", "Multi-seed summary")
    save_table(df_multiseed_tests, "multiseed_paired_tests", "Paired tests: Grid vs Dynamic")
else:
    df_multiseed = pd.DataFrame()
    df_multiseed_summary = pd.DataFrame()
    df_multiseed_tests = pd.DataFrame()

# ==========================================================
# BLOCK 32 — FINAL ARTIFACTS
# ==========================================================

final_artifacts = {
    "mnist_main": df_mnist_main,
    "mnist_dsi_weight_sensitivity": df_mnist_dsi_weight_sensitivity,
    "fashion_search": df_fashion_search,
    "fashion_comparison": df_fashion_comparison,
    "fashion_summary": df_fashion_summary,
    "fashion_sensitivity": df_fashion_sensitivity,
    "fashion_sensitivity_downstream": df_fashion_sensitivity_downstream,
    "fashion_correlation": df_fashion_corr,
    "cifar10_cnn": df_cifar10_cnn,
    "cifar10_resnet_like": df_cifar10_resnet,
    "cifar100_cnn": df_cifar100_cnn,
    "multidataset": df_multidataset,
    "multiseed_summary": df_multiseed_summary,
    "multiseed_tests": df_multiseed_tests
}

save_pickle(final_artifacts, "final_artifacts_for_paper.pkl")
save_json(
    {
        "project_root": PROJECT_ROOT,
        "output_dir": CONFIG.output_dir,
        "data_dir": CONFIG.data_dir
    },
    "project_paths.json"
)

print("\nFull process completed.")
print("All outputs were saved in:", CONFIG.output_dir)

Mounted at /content/drive
Drive mounted successfully.
PROJECT_ROOT = /content/drive/MyDrive/dynamic_stability_lr_project
OUTPUT_ROOT  = /content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs
DATA_ROOT    = /content/drive/MyDrive/dynamic_stability_lr_project/data_cache
CONFIG.data_dir   = /content/drive/MyDrive/dynamic_stability_lr_project/data_cache
CONFIG.output_dir = /content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs
Global seed: 42
Device: cuda
Global learning-rate grid:
[1.00000000e-04 2.31012970e-04 5.33669923e-04 1.23284674e-03
 2.84803587e-03 6.57933225e-03 1.51991108e-02 3.51119173e-02
 8.11130831e-02 1.87381742e-01 4.32876128e-01 1.00000000e+00]

=== MAIN EXPERIMENT: MNIST (MLP) ===



100%|██████████| 9.91M/9.91M [00:01<00:00, 4.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.94MB/s]


,learning_rate,optimizer,scheduler,dsi,volatility,oscillation,descent,instability_penalty,original_proxy,ablation_only_volatility,ablation_only_oscillation,ablation_only_descent_reward,ablation_volatility_plus_oscillation,final_loss,n_steps,has_non_finite,diverged,batches_processed,elapsed_seconds,val_accuracy
0,0.000100,sgd,none,0.493939,0.003706,0.652406,-0.000743,0.0,-5.156510,0.003706,0.652406,0.000743,0.656112,2.306959,189,False,False,189,4.204127,0.1185
1,0.000231,sgd,none,0.493888,0.003700,0.652406,-0.000707,0.0,-5.158185,0.003700,0.652406,0.000707,0.656106,2.306875,189,False,False,189,3.477062,0.1185
2,0.000534,sgd,none,0.493771,0.003687,0.652406,-0.000623,0.0,-5.162091,0.003687,0.652406,0.000623,0.656093,2.306684,189,False,False,189,3.667973,0.1185
3,0.001233,sgd,none,0.493506,0.003657,0.652406,-0.000435,0.0,-5.171660,0.003657,0.652406,0.000435,0.656064,2.306250,189,False,False,189,4.664526,0.1185
4,0.002848,sgd,none,0.492922,0.003590,0.652406,-0.000022,0.0,-5.196792,0.003590,0.652406,0.000022,0.655997,2.305296,189,False,False,189,3.479049,0.1185
5,0.006579,sgd,none,0.491654,0.003442,0.652406,0.000875,0.0,-5.261524,0.003442,0.652406,-0.000875,0.655849,2.303230,189,False,False,189,3.427460,0.1185
6,0.015199,sgd,none,0.488918,0.003151,0.652406,0.002830,0.0,-5.418862,0.003151,0.652406,-0.002830,0.655557,2.298723,189,False,False,189,4.312618,0.1185
7,0.035112,sgd,none,0.489958,0.002708,0.663102,0.008061,0.0,-5.549415,0.002708,0.663102,-0.008061,0.665809,2.286664,189,False,False,189,4.284569,0.1185
8,0.081113,sgd,none,0.326467,0.003164,0.641711,0.126384,0.0,-5.425498,0.003164,0.641711,-0.126384,0.644875,2.013899,189,False,False,189,3.514565,0.3915
9,0.187382,sgd,none,-0.312372,0.056375,0.518717,0.606227,0.0,-3.702273,0.056375,0.518717,-0.606227,0.575091,0.907743,189,False,False,189,3.464148,0.6320


,alpha,beta,gamma,delta,rule_name,quantile,selected_lr,configs_considered
0,0.5,0.5,1.00,1.0,quantile_plus_descent,0.33,0.432876,4
1,0.5,0.5,1.00,2.0,quantile_plus_descent,0.33,0.432876,4
2,0.5,0.5,1.00,3.0,quantile_plus_descent,0.33,0.432876,4
3,0.5,0.5,1.25,1.0,quantile_plus_descent,0.33,0.432876,4
4,0.5,0.5,1.25,2.0,quantile_plus_descent,0.33,0.432876,4



=== METHOD COMPARISON: FASHIONMNIST ===



...

,dataset,architecture,dynamic_rule_name,dynamic_quantile,grid_lr,dynamic_lr,early_lr,random_lr,hybrid_lr,grid_test_accuracy,dynamic_test_accuracy,early_test_accuracy,random_test_accuracy,grid_search_time_seconds,dynamic_search_time_seconds
0,MNIST,mlp,quantile_plus_descent,0.33,0.005995,0.002154,0.005995,0.005995,0.002154,0.944000,0.947200,0.944000,0.944000,14.667933,6.888002
1,FashionMNIST,mlp,quantile_plus_descent,0.33,0.005995,0.005995,0.005995,0.005995,0.000774,0.848700,0.848700,0.848700,0.848700,14.563843,6.056114
2,wine,tabular_mlp,quantile_plus_descent,0.33,0.046416,0.046416,0.129155,0.046416,0.046416,0.925926,0.925926,0.981481,0.925926,2.082662,2.144557
3,breastcancer,tabular_mlp,quantile_plus_descent,0.33,0.016681,0.046416,0.046416,0.016681,0.016681,0.970760,0.959064,0.959064,0.970760,3.210954,1.926075
4,digits,tabular_mlp,quantile_plus_descent,0.33,0.005995,0.016681,0.046416,0.005995,0.016681,0.975926,0.966667,0.914815,0.975926,4.783633,3.403156



=== MULTI-SEED VALIDATION ===

[MULTI-SEED] Seed 7
[MULTI-SEED] Seed=7 | Dataset=MNIST
[MNIST | mlp] search_batches=38 | dynamic_epochs=1
[MULTI-SEED] Seed=7 | Dataset=FashionMNIST
[FashionMNIST | mlp] search_batches=38 | dynamic_epochs=1
[MULTI-SEED] Seed=7 | Dataset=wine
[wine | tabular_mlp] search_batches=4 | dynamic_epochs=2
[MULTI-SEED] Seed=7 | Dataset=breastcancer
[breastcancer | tabular_mlp] search_batches=10 | dynamic_epochs=1
[MULTI-SEED] Seed=7 | Dataset=digits
[digits | tabular_mlp] search_batches=32 | dynamic_epochs=1
[MULTI-SEED] Seed 21
[MULTI-SEED] Seed=21 | Dataset=MNIST
[MNIST | mlp] search_batches=38 | dynamic_epochs=1
[MULTI-SEED] Seed=21 | Dataset=FashionMNIST
[FashionMNIST | mlp] search_batches=38 | dynamic_epochs=1
[MULTI-SEED] Seed=21 | Dataset=wine
[wine | tabular_mlp] search_batches=4 | dynamic_epochs=2
[MULTI-SEED] Seed=21 | Dataset=breastcancer
[breastcancer | tabular_mlp] search_batches=10 | dynamic_epochs=1
[MULTI-SEED] Seed=21 | Dataset=digits
[digits | 

,dataset,architecture,grid_test_accuracy_mean,grid_test_accuracy_std,dynamic_test_accuracy_mean,dynamic_test_accuracy_std,random_test_accuracy_mean,random_test_accuracy_std,grid_search_time_mean,grid_search_time_std,dynamic_search_time_mean,dynamic_search_time_std,dynamic_acc_ci95,grid_acc_ci95
0,FashionMNIST,mlp,0.8377,0.0099,0.8380,0.0090,0.8318,0.0145,14.8316,0.3448,6.6641,0.7154,0.0079,0.0087
1,MNIST,mlp,0.9411,0.0106,0.9387,0.0096,0.9364,0.0111,15.1262,0.4185,6.6466,0.5961,0.0084,0.0093
2,breastcancer,tabular_mlp,0.9801,0.0089,0.9789,0.0128,0.9602,0.0402,2.8204,0.3267,1.6616,0.3822,0.0112,0.0078
3,digits,tabular_mlp,0.9626,0.0127,0.9585,0.0104,0.9556,0.0183,4.9589,0.5031,2.7492,0.4288,0.0091,0.0111
4,wine,tabular_mlp,0.9556,0.0248,0.9333,0.0101,0.9593,0.0241,2.2551,0.2711,2.4193,0.3635,0.0089,0.0218


,dataset,architecture,n_pairs,ttest_pvalue,wilcoxon_pvalue,cohen_d_paired
0,FashionMNIST,mlp,5,0.959960,1.000,-0.023889
1,MNIST,mlp,5,0.255889,0.375,0.592415
2,breastcancer,tabular_mlp,5,0.704000,1.000,0.182574
3,digits,tabular_mlp,5,0.180152,0.500,0.725319
4,wine,tabular_mlp,5,0.177808,0.500,0.730297



Full process completed.
All outputs were saved in: /content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs


In [2]:
# ==========================================================
# EXTRA AUTONOMOUS BLOCK — FROM GOOGLE DRIVE
# Paired tests + WTL + DSI sensitivity + final paper tables
# Does NOT require running the master notebook
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon
from IPython.display import display

# ----------------------------------------------------------
# 0) PERSISTENT OUTPUT PATH IN DRIVE
# ----------------------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using OUTPUT_DIR =", OUTPUT_DIR)

# ----------------------------------------------------------
# 1) REQUIRED INPUT FILES
# ----------------------------------------------------------
required_files = [
    "multiseed_all_runs.csv",
    "mnist_dsi_weight_sensitivity.csv",
    "mnist_main_results.csv"
]

missing_files = [f for f in required_files if not os.path.exists(os.path.join(OUTPUT_DIR, f))]

if missing_files:
    raise FileNotFoundError(
        "Missing required files in Drive. "
        f"Could not find: {missing_files}"
    )

# ----------------------------------------------------------
# 2) GENERAL HELPERS
# ----------------------------------------------------------
PRACTICAL_TIE_EPS = 0.005  # 0.5 percentage points

def cohen_d_paired(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan

    diff = x - y
    std_diff = np.std(diff, ddof=1)

    if std_diff == 0 or np.isnan(std_diff):
        return np.nan

    return float(np.mean(diff) / std_diff)

def outcome_counts(diff_series, eps: float = PRACTICAL_TIE_EPS):
    diff_series = np.asarray(diff_series, dtype=float)
    diff_series = diff_series[np.isfinite(diff_series)]

    wins = int((diff_series > eps).sum())
    ties = int((np.abs(diff_series) <= eps).sum())
    losses = int((diff_series < -eps).sum())

    return wins, ties, losses

def fmt_mean_std(x: np.ndarray, decimals: int = 4) -> str:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return "NaN"
    if len(x) == 1:
        return f"{x.mean():.{decimals}f} ± 0.0000"

    return f"{x.mean():.{decimals}f} ± {x.std(ddof=1):.{decimals}f}"

def safe_paired_tests(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return {
            "n_pairs": len(x),
            "mean_diff": np.nan,
            "ttest_pvalue": np.nan,
            "wilcoxon_pvalue": np.nan,
            "cohen_d_paired": np.nan,
            "wins": np.nan,
            "ties": np.nan,
            "losses": np.nan
        }

    diff = x - y

    try:
        t_p = ttest_rel(x, y, nan_policy="omit").pvalue
    except Exception:
        t_p = np.nan

    try:
        w_p = wilcoxon(x, y).pvalue
    except Exception:
        w_p = np.nan

    wins, ties, losses = outcome_counts(diff)

    return {
        "n_pairs": len(x),
        "mean_diff": float(np.mean(diff)),
        "ttest_pvalue": t_p,
        "wilcoxon_pvalue": w_p,
        "cohen_d_paired": cohen_d_paired(x, y),
        "wins": wins,
        "ties": ties,
        "losses": losses
    }

def export_dataframe_html(df: pd.DataFrame, title: str, filepath: str) -> None:
    html_content = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>{title}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: white;
                color: black;
            }}
            h2 {{
                text-align: center;
            }}
            table {{
                border-collapse: collapse;
                width: 98%;
                margin: auto;
                font-size: 13px;
            }}
            th, td {{
                border: 1px solid #999;
                padding: 6px 8px;
                text-align: center;
            }}
            th {{
                background-color: #f2f2f2;
            }}
        </style>
    </head>
    <body>
        <h2>{title}</h2>
        {df.to_html(index=False)}
    </body>
    </html>
    """
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(html_content)

# ----------------------------------------------------------
# 3) LOAD BASE RESULTS FROM DRIVE
# ----------------------------------------------------------
df_multiseed_runs = pd.read_csv(os.path.join(OUTPUT_DIR, "multiseed_all_runs.csv"))
df_weights = pd.read_csv(os.path.join(OUTPUT_DIR, "mnist_dsi_weight_sensitivity.csv"))
df_mnist_main = pd.read_csv(os.path.join(OUTPUT_DIR, "mnist_main_results.csv"))

print("\nBase files loaded successfully:")
print("- multiseed_all_runs.csv")
print("- mnist_dsi_weight_sensitivity.csv")
print("- mnist_main_results.csv")

# ==========================================================
# EXTRA BLOCK 1 — PAIRED TESTS: DYNAMIC VS RANDOM SEARCH
# ==========================================================
rows = []

for (dataset, architecture), group in df_multiseed_runs.groupby(["dataset", "architecture"]):
    dyn = group["dynamic_test_accuracy"].to_numpy(dtype=float)
    rnd = group["random_test_accuracy"].to_numpy(dtype=float)

    mask = np.isfinite(dyn) & np.isfinite(rnd)
    dyn = dyn[mask]
    rnd = rnd[mask]

    if len(dyn) < 2:
        rows.append({
            "dataset": dataset,
            "architecture": architecture,
            "n_pairs": len(dyn),
            "dynamic_mean": np.nanmean(dyn) if len(dyn) > 0 else np.nan,
            "random_mean": np.nanmean(rnd) if len(rnd) > 0 else np.nan,
            "mean_diff_dynamic_minus_random": np.nan,
            "ttest_pvalue": np.nan,
            "wilcoxon_pvalue": np.nan,
            "cohen_d_paired": np.nan
        })
        continue

    try:
        t_p = ttest_rel(dyn, rnd, nan_policy="omit").pvalue
    except Exception:
        t_p = np.nan

    try:
        w_p = wilcoxon(dyn, rnd).pvalue
    except Exception:
        w_p = np.nan

    rows.append({
        "dataset": dataset,
        "architecture": architecture,
        "n_pairs": len(dyn),
        "dynamic_mean": float(np.mean(dyn)),
        "random_mean": float(np.mean(rnd)),
        "mean_diff_dynamic_minus_random": float(np.mean(dyn - rnd)),
        "ttest_pvalue": t_p,
        "wilcoxon_pvalue": w_p,
        "cohen_d_paired": cohen_d_paired(dyn, rnd)
    })

df_dynamic_vs_random_tests = (
    pd.DataFrame(rows)
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print("\n=== EXTRA 1 — Dynamic vs Random ===\n")
display(df_dynamic_vs_random_tests)

df_dynamic_vs_random_tests.to_csv(
    os.path.join(OUTPUT_DIR, "extra_dynamic_vs_random_tests.csv"),
    index=False
)

# ==========================================================
# EXTRA BLOCK 2 — WIN / TIE / LOSS BY DATASET
# ==========================================================
rows = []

for (dataset, architecture), group in df_multiseed_runs.groupby(["dataset", "architecture"]):
    diff_dyn_grid = group["dynamic_test_accuracy"] - group["grid_test_accuracy"]
    diff_dyn_random = group["dynamic_test_accuracy"] - group["random_test_accuracy"]

    w1, t1, l1 = outcome_counts(diff_dyn_grid)
    w2, t2, l2 = outcome_counts(diff_dyn_random)

    rows.append({
        "dataset": dataset,
        "architecture": architecture,
        "dyn_minus_grid_mean": float(diff_dyn_grid.mean()),
        "dyn_minus_grid_std": float(diff_dyn_grid.std(ddof=1)),
        "dyn_vs_grid_wins": w1,
        "dyn_vs_grid_ties": t1,
        "dyn_vs_grid_losses": l1,
        "dyn_minus_random_mean": float(diff_dyn_random.mean()),
        "dyn_minus_random_std": float(diff_dyn_random.std(ddof=1)),
        "dyn_vs_random_wins": w2,
        "dyn_vs_random_ties": t2,
        "dyn_vs_random_losses": l2
    })

df_win_tie_loss = (
    pd.DataFrame(rows)
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print("\n=== EXTRA 2 — Win / Tie / Loss ===\n")
display(df_win_tie_loss)

df_win_tie_loss.to_csv(
    os.path.join(OUTPUT_DIR, "extra_win_tie_loss.csv"),
    index=False
)

# ==========================================================
# EXTRA BLOCK 3 — DSI WEIGHT SENSITIVITY + ACCURACY
# ==========================================================
df_weight_accuracy = df_weights.merge(
    df_mnist_main[["learning_rate", "val_accuracy", "dsi", "descent"]],
    left_on="selected_lr",
    right_on="learning_rate",
    how="left"
).drop(columns=["learning_rate"])

df_lr_frequency = (
    df_weight_accuracy.groupby("selected_lr", dropna=False)
    .size()
    .reset_index(name="times_selected")
    .sort_values(["times_selected", "selected_lr"], ascending=[False, True])
    .reset_index(drop=True)
)

print("\n=== EXTRA 3 — DSI weight sensitivity ===\n")
display(df_weight_accuracy.head(10))
display(df_lr_frequency)

df_weight_accuracy.to_csv(
    os.path.join(OUTPUT_DIR, "extra_dsi_weights_with_accuracy.csv"),
    index=False
)

df_lr_frequency.to_csv(
    os.path.join(OUTPUT_DIR, "extra_dsi_selected_lr_frequency.csv"),
    index=False
)

# ==========================================================
# EXTRA BLOCK 4 — FINAL PAPER-STYLE TABLE
# ==========================================================
rows = []

for (dataset, architecture), group in df_multiseed_runs.groupby(["dataset", "architecture"]):
    dyn = group["dynamic_test_accuracy"].to_numpy(dtype=float)
    grd = group["grid_test_accuracy"].to_numpy(dtype=float)
    rnd = group["random_test_accuracy"].to_numpy(dtype=float)

    dg_stats = safe_paired_tests(dyn, grd)
    dr_stats = safe_paired_tests(dyn, rnd)

    rows.append({
        "dataset": dataset,
        "architecture": architecture,

        "dynamic_mean_std": fmt_mean_std(dyn),
        "grid_mean_std": fmt_mean_std(grd),
        "random_mean_std": fmt_mean_std(rnd),

        "dyn_minus_grid_mean_diff": dg_stats["mean_diff"],
        "dyn_vs_grid_ttest_p": dg_stats["ttest_pvalue"],
        "dyn_vs_grid_wilcoxon_p": dg_stats["wilcoxon_pvalue"],
        "dyn_vs_grid_cohen_d": dg_stats["cohen_d_paired"],
        "dyn_vs_grid_wtl": f'{dg_stats["wins"]}/{dg_stats["ties"]}/{dg_stats["losses"]}',

        "dyn_minus_random_mean_diff": dr_stats["mean_diff"],
        "dyn_vs_random_ttest_p": dr_stats["ttest_pvalue"],
        "dyn_vs_random_wilcoxon_p": dr_stats["wilcoxon_pvalue"],
        "dyn_vs_random_cohen_d": dr_stats["cohen_d_paired"],
        "dyn_vs_random_wtl": f'{dr_stats["wins"]}/{dr_stats["ties"]}/{dr_stats["losses"]}',

        "n_pairs": dg_stats["n_pairs"]
    })

df_paper_summary = (
    pd.DataFrame(rows)
    .sort_values(["dataset", "architecture"])
    .reset_index(drop=True)
)

print("\n=== EXTRA 4 — Final paper-style table ===\n")
display(df_paper_summary)

df_paper_summary.to_csv(
    os.path.join(OUTPUT_DIR, "extra_paper_summary.csv"),
    index=False
)

df_paper_summary_compact = df_paper_summary[[
    "dataset",
    "architecture",
    "dynamic_mean_std",
    "grid_mean_std",
    "random_mean_std",
    "dyn_minus_grid_mean_diff",
    "dyn_vs_grid_ttest_p",
    "dyn_vs_grid_cohen_d",
    "dyn_vs_grid_wtl",
    "dyn_minus_random_mean_diff",
    "dyn_vs_random_ttest_p",
    "dyn_vs_random_cohen_d",
    "dyn_vs_random_wtl"
]].copy()

display(df_paper_summary_compact)

df_paper_summary_compact.to_csv(
    os.path.join(OUTPUT_DIR, "extra_paper_summary_compact.csv"),
    index=False
)

export_dataframe_html(
    df_paper_summary,
    "Final summary table for paper",
    os.path.join(OUTPUT_DIR, "extra_paper_summary.html")
)

export_dataframe_html(
    df_paper_summary_compact,
    "Compact final summary table for paper",
    os.path.join(OUTPUT_DIR, "extra_paper_summary_compact.html")
)

# ----------------------------------------------------------
# 4) FINAL SUMMARY
# ----------------------------------------------------------
print("\nFinal paper-style outputs saved at:")
print(os.path.join(OUTPUT_DIR, "extra_dynamic_vs_random_tests.csv"))
print(os.path.join(OUTPUT_DIR, "extra_win_tie_loss.csv"))
print(os.path.join(OUTPUT_DIR, "extra_dsi_weights_with_accuracy.csv"))
print(os.path.join(OUTPUT_DIR, "extra_dsi_selected_lr_frequency.csv"))
print(os.path.join(OUTPUT_DIR, "extra_paper_summary.csv"))
print(os.path.join(OUTPUT_DIR, "extra_paper_summary_compact.csv"))
print(os.path.join(OUTPUT_DIR, "extra_paper_summary.html"))
print(os.path.join(OUTPUT_DIR, "extra_paper_summary_compact.html"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using OUTPUT_DIR = /content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs

Base files loaded successfully:
- multiseed_all_runs.csv
- mnist_dsi_weight_sensitivity.csv
- mnist_main_results.csv

=== EXTRA 1 — Dynamic vs Random ===



,dataset,architecture,n_pairs,dynamic_mean,random_mean,mean_diff_dynamic_minus_random,ttest_pvalue,wilcoxon_pvalue,cohen_d_paired
0,FashionMNIST,mlp,5,0.837980,0.831820,0.006160,0.178240,0.50,0.729375
1,MNIST,mlp,5,0.938680,0.936400,0.002280,0.386682,0.50,0.434093
2,breastcancer,tabular_mlp,5,0.978947,0.960234,0.018713,0.438199,1.00,0.384678
3,digits,tabular_mlp,5,0.958519,0.955556,0.002963,0.623177,1.00,0.237723
4,wine,tabular_mlp,5,0.933333,0.959259,-0.025926,0.079960,0.25,-1.043498



=== EXTRA 2 — Win / Tie / Loss ===



,dataset,architecture,dyn_minus_grid_mean,dyn_minus_grid_std,dyn_vs_grid_wins,dyn_vs_grid_ties,dyn_vs_grid_losses,dyn_minus_random_mean,dyn_minus_random_std,dyn_vs_random_wins,dyn_vs_random_ties,dyn_vs_random_losses
0,FashionMNIST,mlp,0.000240,0.010046,1,3,1,0.006160,0.008446,2,3,0
1,MNIST,mlp,-0.002420,0.004085,0,3,2,0.002280,0.005252,1,4,0
2,breastcancer,tabular_mlp,-0.001170,0.006406,1,3,1,0.018713,0.048647,1,3,1
3,digits,tabular_mlp,-0.004074,0.005617,0,3,2,0.002963,0.012464,1,3,1
4,wine,tabular_mlp,-0.022222,0.030429,0,3,2,-0.025926,0.024845,0,2,3



=== EXTRA 3 — DSI weight sensitivity ===



...

,selected_lr,times_selected
0,0.432876,81



=== EXTRA 4 — Final paper-style table ===



,dataset,architecture,dynamic_mean_std,grid_mean_std,random_mean_std,dyn_minus_grid_mean_diff,dyn_vs_grid_ttest_p,dyn_vs_grid_wilcoxon_p,dyn_vs_grid_cohen_d,dyn_vs_grid_wtl,dyn_minus_random_mean_diff,dyn_vs_random_ttest_p,dyn_vs_random_wilcoxon_p,dyn_vs_random_cohen_d,dyn_vs_random_wtl,n_pairs
0,FashionMNIST,mlp,0.8380 ± 0.0090,0.8377 ± 0.0099,0.8318 ± 0.0145,0.000240,0.959960,1.000,0.023889,1/3/1,0.006160,0.178240,0.50,0.729375,2/3/0,5
1,MNIST,mlp,0.9387 ± 0.0096,0.9411 ± 0.0106,0.9364 ± 0.0111,-0.002420,0.255889,0.375,-0.592415,0/3/2,0.002280,0.386682,0.50,0.434093,1/4/0,5
2,breastcancer,tabular_mlp,0.9789 ± 0.0128,0.9801 ± 0.0089,0.9602 ± 0.0402,-0.001170,0.704000,1.000,-0.182574,1/3/1,0.018713,0.438199,1.00,0.384678,1/3/1,5
3,digits,tabular_mlp,0.9585 ± 0.0104,0.9626 ± 0.0127,0.9556 ± 0.0183,-0.004074,0.180152,0.500,-0.725319,0/3/2,0.002963,0.623177,1.00,0.237723,1/3/1,5
4,wine,tabular_mlp,0.9333 ± 0.0101,0.9556 ± 0.0248,0.9593 ± 0.0241,-0.022222,0.177808,0.500,-0.730297,0/3/2,-0.025926,0.079960,0.25,-1.043498,0/2/3,5


,dataset,architecture,dynamic_mean_std,grid_mean_std,random_mean_std,dyn_minus_grid_mean_diff,dyn_vs_grid_ttest_p,dyn_vs_grid_cohen_d,dyn_vs_grid_wtl,dyn_minus_random_mean_diff,dyn_vs_random_ttest_p,dyn_vs_random_cohen_d,dyn_vs_random_wtl
0,FashionMNIST,mlp,0.8380 ± 0.0090,0.8377 ± 0.0099,0.8318 ± 0.0145,0.000240,0.959960,0.023889,1/3/1,0.006160,0.178240,0.729375,2/3/0
1,MNIST,mlp,0.9387 ± 0.0096,0.9411 ± 0.0106,0.9364 ± 0.0111,-0.002420,0.255889,-0.592415,0/3/2,0.002280,0.386682,0.434093,1/4/0
2,breastcancer,tabular_mlp,0.9789 ± 0.0128,0.9801 ± 0.0089,0.9602 ± 0.0402,-0.001170,0.704000,-0.182574,1/3/1,0.018713,0.438199,0.384678,1/3/1
3,digits,tabular_mlp,0.9585 ± 0.0104,0.9626 ± 0.0127,0.9556 ± 0.0183,-0.004074,0.180152,-0.725319,0/3/2,0.002963,0.623177,0.237723,1/3/1
4,wine,tabular_mlp,0.9333 ± 0.0101,0.9556 ± 0.0248,0.9593 ± 0.0241,-0.022222,0.177808,-0.730297,0/3/2,-0.025926,0.079960,-1.043498,0/2/3



Final paper-style outputs saved at:
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_dynamic_vs_random_tests.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_win_tie_loss.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_dsi_weights_with_accuracy.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_dsi_selected_lr_frequency.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_paper_summary.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_paper_summary_compact.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_paper_summary.html
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/extra_paper_summary_compact.html


In [3]:
# ==========================================================
# SINGLE NOTEBOOK — RAISIN + NASA-CMAPSS WITH DSI
# Compact real-world applications of the Dynamic Stability Index
# + final comparison table + confusion matrices
# ==========================================================

# ----------------------------------------------------------
# 0) DRIVE + LIBRARIES
# ----------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import time
import math
import random
import subprocess
from dataclasses import dataclass, asdict
from typing import Dict, List, Any, Optional, Callable, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.express as px

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from IPython.display import display

# Install dependencies if missing
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])
    from ucimlrepo import fetch_ucirepo

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

# ----------------------------------------------------------
# 1) GLOBAL CONFIGURATION
# ----------------------------------------------------------
@dataclass
class MultiRealDSIConfig:
    seed: int = 42

    root_dir: str = "/content/drive/MyDrive/dynamic_stability_lr_project"
    output_dir_name: str = "paper_outputs"
    data_dir_name: str = "data_cache"

    batch_size_train: int = 64
    batch_size_eval: int = 128
    num_workers: int = 2
    pin_memory: bool = True

    lr_min: float = 1e-4
    lr_max: float = 1.0
    lr_points: int = 10

    exploratory_epochs: int = 3
    final_epochs_raisin: int = 20
    final_epochs_nasa: int = 12

    dynamic_search_max_batches: int = 30
    early_loss_max_batches: int = 16
    random_search_samples: int = 4

    eps: float = 1e-8
    min_series_points: int = 8

    dsi_alpha_volatility: float = 1.0
    dsi_beta_oscillation: float = 0.75
    dsi_gamma_descent: float = 1.25
    dsi_delta_instability: float = 2.0

    nasa_risk_horizon_rul: int = 30

CONFIG = MultiRealDSIConfig()

OUTPUT_DIR = os.path.join(CONFIG.root_dir, CONFIG.output_dir_name)
DATA_DIR = os.path.join(CONFIG.root_dir, CONFIG.data_dir_name)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LEARNING_RATES = np.logspace(
    np.log10(CONFIG.lr_min),
    np.log10(CONFIG.lr_max),
    CONFIG.lr_points
)

print("OUTPUT_DIR =", OUTPUT_DIR)
print("DATA_DIR   =", DATA_DIR)
print("DEVICE     =", DEVICE)
print("Learning rates =", LEARNING_RATES)

# ----------------------------------------------------------
# 2) REPRODUCIBILITY
# ----------------------------------------------------------
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_global_seed(CONFIG.seed)

# ----------------------------------------------------------
# 3) GENERAL HELPERS
# ----------------------------------------------------------
def make_loader(dataset, batch_size: int, shuffle: bool, seed: Optional[int] = None) -> DataLoader:
    generator = None
    if shuffle and seed is not None:
        generator = torch.Generator().manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CONFIG.num_workers,
        pin_memory=CONFIG.pin_memory if DEVICE.type == "cuda" else False,
        worker_init_fn=seed_worker,
        generator=generator
    )
    loader._experiment_shuffle = shuffle
    return loader

def rebuild_loader_with_seed(loader: DataLoader, seed: int) -> DataLoader:
    shuffle = getattr(loader, "_experiment_shuffle", False)
    generator = torch.Generator().manual_seed(seed) if shuffle else None

    cloned_loader = DataLoader(
        loader.dataset,
        batch_size=loader.batch_size,
        shuffle=shuffle,
        num_workers=loader.num_workers,
        pin_memory=loader.pin_memory,
        drop_last=loader.drop_last,
        collate_fn=loader.collate_fn,
        worker_init_fn=seed_worker,
        generator=generator
    )
    cloned_loader._experiment_shuffle = shuffle
    return cloned_loader

def sanitize_series(values: List[float]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return arr

def has_non_finite(values: List[float]) -> bool:
    arr = np.asarray(values, dtype=float)
    return not np.all(np.isfinite(arr))

def safe_mean(values: List[float]) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) > 0 else np.nan

def safe_divide(numerator: float, denominator: float) -> float:
    if numerator is None or denominator is None:
        return np.nan
    if not np.isfinite(numerator) or not np.isfinite(denominator):
        return np.nan
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)

def export_dataframe_html(df: pd.DataFrame, title: str, filepath: str) -> None:
    html_content = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>{title}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: white;
                color: black;
            }}
            h2 {{
                text-align: center;
            }}
            table {{
                border-collapse: collapse;
                width: 98%;
                margin: auto;
                font-size: 13px;
            }}
            th, td {{
                border: 1px solid #999;
                padding: 6px 8px;
                text-align: center;
            }}
            th {{
                background-color: #f2f2f2;
            }}
        </style>
    </head>
    <body>
        <h2>{title}</h2>
        {df.to_html(index=False)}
    </body>
    </html>
    """
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(html_content)

# ----------------------------------------------------------
# 4) LIGHTWEIGHT TABULAR MODEL
# ----------------------------------------------------------
class TabularMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim)
        )

    def forward(self, x):
        return self.net(x)

def build_optimizer(optimizer_name: str, params, lr: float):
    optimizer_name = optimizer_name.lower()

    if optimizer_name == "adam":
        return optim.Adam(params, lr=lr)
    elif optimizer_name == "sgd":
        return optim.SGD(params, lr=lr)
    elif optimizer_name == "rmsprop":
        return optim.RMSprop(params, lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

def build_scheduler(scheduler_name: str, optimizer, total_epochs: int):
    scheduler_name = scheduler_name.lower()

    if scheduler_name == "none":
        return None
    elif scheduler_name == "steplr":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=max(1, total_epochs // 2),
            gamma=0.5
        )
    elif scheduler_name == "cosineannealinglr":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=total_epochs
        )
    else:
        raise ValueError(f"Unsupported scheduler: {scheduler_name}")

# ----------------------------------------------------------
# 5) NORMS AND CHECKS
# ----------------------------------------------------------
def compute_global_weight_norm(model: nn.Module) -> float:
    sq_sum = 0.0
    for p in model.parameters():
        sq_sum += p.data.pow(2).sum().item()
    return math.sqrt(sq_sum)

def compute_global_grad_norm(model: nn.Module) -> float:
    sq_sum = 0.0
    for p in model.parameters():
        if p.grad is not None:
            sq_sum += p.grad.pow(2).sum().item()
    return math.sqrt(sq_sum)

def model_has_non_finite_params(model: nn.Module) -> bool:
    for p in model.parameters():
        if not torch.isfinite(p.data).all():
            return True
    return False

def model_has_non_finite_grads(model: nn.Module) -> bool:
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            return True
    return False

# ----------------------------------------------------------
# 6) DSI
# ----------------------------------------------------------
def empirical_loss_volatility(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)
    if len(clean) < CONFIG.min_series_points:
        return np.nan
    rel_diffs = np.abs(np.diff(clean)) / (np.abs(clean[:-1]) + eps)
    if len(rel_diffs) == 0:
        return np.nan
    return float(np.mean(np.log1p(rel_diffs)))

def empirical_loss_oscillation(losses: List[float]) -> float:
    clean = sanitize_series(losses)
    if len(clean) < CONFIG.min_series_points:
        return np.nan
    deltas = np.diff(clean)
    if len(deltas) < 2:
        return np.nan
    sign_changes = (deltas[1:] * deltas[:-1] < 0).astype(float)
    if len(sign_changes) == 0:
        return np.nan
    return float(np.mean(sign_changes))

def normalized_loss_descent(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)
    if len(clean) < 2:
        return np.nan
    initial = clean[0]
    final = clean[-1]
    return float((initial - final) / (abs(initial) + eps))

def instability_penalty(losses: List[float], has_nonfinite_flag: bool = False) -> float:
    if has_nonfinite_flag or has_non_finite(losses):
        return 1.0
    return 0.0

def dynamic_stability_index(
    losses: List[float],
    has_nonfinite_flag: bool = False,
    alpha: float = CONFIG.dsi_alpha_volatility,
    beta: float = CONFIG.dsi_beta_oscillation,
    gamma: float = CONFIG.dsi_gamma_descent,
    delta: float = CONFIG.dsi_delta_instability
) -> Dict[str, float]:
    volatility = empirical_loss_volatility(losses)
    oscillation = empirical_loss_oscillation(losses)
    descent = normalized_loss_descent(losses)
    penalty = instability_penalty(losses, has_nonfinite_flag)

    if penalty > 0:
        total_dsi = np.inf
    elif np.isnan(volatility) or np.isnan(oscillation) or np.isnan(descent):
        total_dsi = np.nan
    else:
        total_dsi = alpha * volatility + beta * oscillation - gamma * descent + delta * penalty

    return {
        "dsi": float(total_dsi) if np.isfinite(total_dsi) else total_dsi,
        "volatility": volatility,
        "oscillation": oscillation,
        "descent": descent,
        "instability_penalty": penalty
    }

def original_log_diff_proxy(losses: List[float], eps: float = CONFIG.eps) -> float:
    clean = sanitize_series(losses)
    if len(clean) < CONFIG.min_series_points:
        return np.nan
    diffs = np.abs(np.diff(clean))
    diffs = diffs[diffs > eps]
    if len(diffs) == 0:
        return np.nan
    return float(np.mean(np.log(diffs)))

def dsi_component_ablation(
    losses: List[float],
    has_nonfinite_flag: bool = False,
    alpha: float = CONFIG.dsi_alpha_volatility,
    beta: float = CONFIG.dsi_beta_oscillation,
    gamma: float = CONFIG.dsi_gamma_descent,
    delta: float = CONFIG.dsi_delta_instability
) -> Dict[str, float]:
    volatility = empirical_loss_volatility(losses)
    oscillation = empirical_loss_oscillation(losses)
    descent = normalized_loss_descent(losses)
    penalty = instability_penalty(losses, has_nonfinite_flag)

    full = dynamic_stability_index(
        losses,
        has_nonfinite_flag=has_nonfinite_flag,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        delta=delta
    )["dsi"]

    return {
        "full_dsi": full,
        "only_volatility": volatility,
        "only_oscillation": oscillation,
        "only_descent_reward": (-descent if not np.isnan(descent) else np.nan),
        "volatility_plus_oscillation": (
            volatility + oscillation if not (np.isnan(volatility) or np.isnan(oscillation)) else np.nan
        ),
        "penalty": penalty
    }

# ----------------------------------------------------------
# 7) EVALUATION
# ----------------------------------------------------------
def evaluate_metrics(model: nn.Module, loader: DataLoader) -> Dict[str, Any]:
    model.eval()

    all_true = []
    all_pred = []
    all_prob1 = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_true.extend(y.numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())
            all_prob1.extend(probs.cpu().numpy().tolist())

    y_true = np.asarray(all_true, dtype=int)
    y_pred = np.asarray(all_pred, dtype=int)
    prob1 = np.asarray(all_prob1, dtype=float)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "prob_class_1": prob1,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_class_1": precision_score(y_true, y_pred, zero_division=0),
        "recall_class_1": recall_score(y_true, y_pred, zero_division=0),
        "f1_class_1": f1_score(y_true, y_pred, zero_division=0)
    }

# ----------------------------------------------------------
# 8) TRAINING
# ----------------------------------------------------------
def run_training_trajectory(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    lr: float,
    class_weights: Optional[torch.Tensor] = None,
    optimizer_name: str = "adam",
    epochs: int = 3,
    record_dynamics: bool = True,
    scheduler_name: str = "none",
    max_batches: Optional[int] = None,
    run_seed: Optional[int] = None
) -> Dict[str, Any]:
    if run_seed is not None:
        set_global_seed(run_seed)
        effective_train_loader = rebuild_loader_with_seed(train_loader, run_seed)
    else:
        effective_train_loader = train_loader

    model = model_builder().to(DEVICE)
    optimizer = build_optimizer(optimizer_name, model.parameters(), lr)
    scheduler = build_scheduler(scheduler_name, optimizer, epochs)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE) if class_weights is not None else None
    )

    loss_series = []
    grad_norm_series = []
    weight_norm_series = []

    numerical_instability = False
    diverged = False
    batches_processed = 0
    start_time = time.perf_counter()

    model.train()

    for epoch in range(epochs):
        for x, y in effective_train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)

            if not torch.isfinite(loss):
                numerical_instability = True
                diverged = True
                break

            loss.backward()

            if model_has_non_finite_grads(model):
                numerical_instability = True
                diverged = True
                break

            if record_dynamics:
                grad_norm_series.append(compute_global_grad_norm(model))

            optimizer.step()
            batches_processed += 1

            if model_has_non_finite_params(model):
                numerical_instability = True
                diverged = True
                break

            if record_dynamics:
                weight_norm_series.append(compute_global_weight_norm(model))
                loss_series.append(loss.item())

            if max_batches is not None and batches_processed >= max_batches:
                break

        if diverged:
            break

        if scheduler is not None:
            scheduler.step()

        if max_batches is not None and batches_processed >= max_batches:
            break

    end_time = time.perf_counter()

    return {
        "model": model,
        "loss_series": loss_series,
        "grad_norm_series": grad_norm_series,
        "weight_norm_series": weight_norm_series,
        "has_non_finite": numerical_instability or has_non_finite(loss_series),
        "diverged": diverged,
        "batches_processed": batches_processed,
        "elapsed_seconds": end_time - start_time
    }

# ----------------------------------------------------------
# 9) SWEEPS
# ----------------------------------------------------------
def run_lr_sweep(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    learning_rates: np.ndarray,
    class_weights: Optional[torch.Tensor] = None,
    optimizer_name: str = "adam",
    epochs: int = 3,
    scheduler_name: str = "none",
    max_batches: Optional[int] = None,
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    rows = []

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            class_weights=class_weights,
            optimizer_name=optimizer_name,
            epochs=epochs,
            record_dynamics=True,
            scheduler_name=scheduler_name,
            max_batches=max_batches,
            run_seed=run_seed
        )

        losses = out["loss_series"]
        dsi_dict = dynamic_stability_index(losses=losses, has_nonfinite_flag=out["has_non_finite"])
        ablation_dict = dsi_component_ablation(losses, has_nonfinite_flag=out["has_non_finite"])
        original_proxy = original_log_diff_proxy(losses)

        rows.append({
            "learning_rate": float(lr),
            "dsi": dsi_dict["dsi"],
            "volatility": dsi_dict["volatility"],
            "oscillation": dsi_dict["oscillation"],
            "descent": dsi_dict["descent"],
            "instability_penalty": dsi_dict["instability_penalty"],
            "original_proxy": original_proxy,
            "ablation_only_volatility": ablation_dict["only_volatility"],
            "ablation_only_oscillation": ablation_dict["only_oscillation"],
            "ablation_only_descent_reward": ablation_dict["only_descent_reward"],
            "ablation_volatility_plus_oscillation": ablation_dict["volatility_plus_oscillation"],
            "final_loss": losses[-1] if len(losses) > 0 else np.nan,
            "n_steps": len(losses),
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"],
            "diverged": out["diverged"]
        })

    return pd.DataFrame(rows).sort_values("learning_rate").reset_index(drop=True)

def run_validation_metric_sweep(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rates: np.ndarray,
    class_weights: Optional[torch.Tensor] = None,
    optimizer_name: str = "adam",
    epochs: int = 3,
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    rows = []

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            class_weights=class_weights,
            optimizer_name=optimizer_name,
            epochs=epochs,
            record_dynamics=False,
            run_seed=run_seed
        )

        if out["diverged"]:
            acc = bal_acc = prec = rec = f1 = np.nan
        else:
            metrics = evaluate_metrics(out["model"], val_loader)
            acc = metrics["accuracy"]
            bal_acc = metrics["balanced_accuracy"]
            prec = metrics["precision_class_1"]
            rec = metrics["recall_class_1"]
            f1 = metrics["f1_class_1"]

        rows.append({
            "learning_rate": float(lr),
            "val_accuracy": acc,
            "val_balanced_accuracy": bal_acc,
            "val_precision_class_1": prec,
            "val_recall_class_1": rec,
            "val_f1_class_1": f1,
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"],
            "diverged": out["diverged"]
        })

    return pd.DataFrame(rows).sort_values("learning_rate").reset_index(drop=True)

def run_early_loss_sweep(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    learning_rates: np.ndarray,
    class_weights: Optional[torch.Tensor] = None,
    optimizer_name: str = "adam",
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    rows = []

    for lr in learning_rates:
        out = run_training_trajectory(
            model_builder=model_builder,
            train_loader=train_loader,
            lr=float(lr),
            class_weights=class_weights,
            optimizer_name=optimizer_name,
            epochs=1,
            record_dynamics=True,
            max_batches=CONFIG.early_loss_max_batches,
            run_seed=run_seed
        )

        rows.append({
            "learning_rate": float(lr),
            "early_loss_mean": np.nan if out["diverged"] else safe_mean(out["loss_series"]),
            "batches_processed": out["batches_processed"],
            "elapsed_seconds": out["elapsed_seconds"],
            "diverged": out["diverged"]
        })

    return pd.DataFrame(rows).sort_values("learning_rate").reset_index(drop=True)

def run_random_search_validation(
    model_builder: Callable[[], nn.Module],
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rates: np.ndarray,
    class_weights: Optional[torch.Tensor] = None,
    optimizer_name: str = "adam",
    epochs: int = CONFIG.exploratory_epochs,
    n_samples: int = CONFIG.random_search_samples,
    sampling_seed: int = CONFIG.seed,
    run_seed: Optional[int] = None
) -> pd.DataFrame:
    unique_lrs = np.asarray(sorted({float(x) for x in learning_rates}))
    rng = np.random.default_rng(sampling_seed)
    sampled_lrs = np.sort(
        rng.choice(unique_lrs, size=min(n_samples, len(unique_lrs)), replace=False)
    )

    df_random = run_validation_metric_sweep(
        model_builder=model_builder,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rates=sampled_lrs,
        class_weights=class_weights,
        optimizer_name=optimizer_name,
        epochs=epochs,
        run_seed=run_seed
    )

    return df_random

# ----------------------------------------------------------
# 10) LR SELECTION
# ----------------------------------------------------------
def select_lr_by_metric(df_val: pd.DataFrame, metric_col: str) -> float:
    df_valid = df_val.dropna(subset=[metric_col]).copy()
    if len(df_valid) == 0:
        raise ValueError(f"No valid configurations for {metric_col}.")
    return float(df_valid.loc[df_valid[metric_col].idxmax(), "learning_rate"])

def select_lr_by_early_loss_sweep(df_early: pd.DataFrame) -> float:
    df_valid = df_early.dropna(subset=["early_loss_mean"]).copy()
    if len(df_valid) == 0:
        raise ValueError("No valid configurations in early-loss sweep.")
    return float(df_valid.loc[df_valid["early_loss_mean"].idxmin(), "learning_rate"])

def dynamic_rule_quantile_plus_descent(df_dynamic: pd.DataFrame, quantile: float = 0.33) -> Tuple[float, int]:
    df_valid = df_dynamic.dropna(subset=["dsi", "descent"]).copy()
    df_valid = df_valid[np.isfinite(df_valid["dsi"])]

    if len(df_valid) == 0:
        raise ValueError("No valid configurations for quantile_plus_descent.")

    threshold = df_valid["dsi"].quantile(quantile)
    region = df_valid[df_valid["dsi"] <= threshold].copy()

    if len(region) == 0:
        best_row = df_valid.loc[df_valid["dsi"].idxmin()]
        return float(best_row["learning_rate"]), 1

    best_row = region.loc[region["descent"].idxmax()]
    return float(best_row["learning_rate"]), len(region)

def resolve_dynamic_epochs_for_dsi(
    train_loader: DataLoader,
    min_points: int = CONFIG.min_series_points,
    max_batches: Optional[int] = CONFIG.dynamic_search_max_batches
) -> int:
    batches_per_epoch = len(train_loader)

    if batches_per_epoch <= 0:
        return 1

    effective_batches_per_epoch = batches_per_epoch
    if max_batches is not None:
        effective_batches_per_epoch = min(effective_batches_per_epoch, max_batches)

    return max(1, math.ceil(min_points / effective_batches_per_epoch))

# ----------------------------------------------------------
# 11) FINAL TRAINING
# ----------------------------------------------------------
def train_final_and_evaluate(
    model_builder: Callable[[], nn.Module],
    train_full_loader: DataLoader,
    test_loader: DataLoader,
    lr: float,
    class_weights: Optional[torch.Tensor] = None,
    epochs: int = 10,
    run_seed: Optional[int] = None
) -> Dict[str, Any]:
    out = run_training_trajectory(
        model_builder=model_builder,
        train_loader=train_full_loader,
        lr=float(lr),
        class_weights=class_weights,
        optimizer_name="adam",
        epochs=epochs,
        record_dynamics=False,
        run_seed=run_seed
    )

    if out["diverged"]:
        return {
            "test_accuracy": np.nan,
            "test_balanced_accuracy": np.nan,
            "test_precision_class_1": np.nan,
            "test_recall_class_1": np.nan,
            "test_f1_class_1": np.nan,
            "train_time_seconds": out["elapsed_seconds"],
            "batches_processed": out["batches_processed"],
            "diverged": True,
            "predictions_df": pd.DataFrame(),
            "y_true": np.array([]),
            "y_pred": np.array([])
        }

    metrics = evaluate_metrics(out["model"], test_loader)

    df_predictions = pd.DataFrame({
        "y_true": metrics["y_true"],
        "y_pred": metrics["y_pred"],
        "prob_pred_class_1": metrics["prob_class_1"]
    })

    return {
        "test_accuracy": metrics["accuracy"],
        "test_balanced_accuracy": metrics["balanced_accuracy"],
        "test_precision_class_1": metrics["precision_class_1"],
        "test_recall_class_1": metrics["recall_class_1"],
        "test_f1_class_1": metrics["f1_class_1"],
        "train_time_seconds": out["elapsed_seconds"],
        "batches_processed": out["batches_processed"],
        "diverged": False,
        "predictions_df": df_predictions,
        "y_true": metrics["y_true"],
        "y_pred": metrics["y_pred"]
    }

# ----------------------------------------------------------
# 12) RAISIN LOADER
# ----------------------------------------------------------
def load_raisin_case(seed: int = CONFIG.seed) -> Dict[str, Any]:
    raisin = fetch_ucirepo(id=850)

    X = raisin.data.features.copy()
    y = raisin.data.targets.copy()

    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]

    y = y.astype(str).str.strip()
    unique_classes = sorted(y.unique().tolist())
    class_to_int = {cls: idx for idx, cls in enumerate(unique_classes)}
    int_to_class = {idx: cls for cls, idx in class_to_int.items()}
    y = y.map(class_to_int).astype(int)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.15, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=(0.15 / 0.85),
        random_state=seed,
        stratify=y_train_full
    )

    scaler_search = StandardScaler()
    X_train_scaled = scaler_search.fit_transform(X_train).astype(np.float32)
    X_val_scaled = scaler_search.transform(X_val).astype(np.float32)

    scaler_final = StandardScaler()
    X_train_full_scaled = scaler_final.fit_transform(X_train_full).astype(np.float32)
    X_test_scaled = scaler_final.transform(X_test).astype(np.float32)

    train_search_ds = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train.to_numpy(), dtype=torch.long)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32),
        torch.tensor(y_val.to_numpy(), dtype=torch.long)
    )
    train_full_ds = TensorDataset(
        torch.tensor(X_train_full_scaled, dtype=torch.float32),
        torch.tensor(y_train_full.to_numpy(), dtype=torch.long)
    )
    test_ds = TensorDataset(
        torch.tensor(X_test_scaled, dtype=torch.float32),
        torch.tensor(y_test.to_numpy(), dtype=torch.long)
    )

    return {
        "dataset_name": "Raisin",
        "positive_class_name": int_to_class[1],
        "selection_metric": "val_accuracy",
        "final_epochs": CONFIG.final_epochs_raisin,
        "train_search_loader": make_loader(train_search_ds, CONFIG.batch_size_train, True, seed=seed),
        "val_loader": make_loader(val_ds, CONFIG.batch_size_eval, False, seed=seed),
        "train_full_loader": make_loader(train_full_ds, CONFIG.batch_size_train, True, seed=seed),
        "test_loader": make_loader(test_ds, CONFIG.batch_size_eval, False, seed=seed),
        "search_class_weights": None,
        "final_class_weights": None,
        "model_builder": lambda: TabularMLP(input_dim=X_train_scaled.shape[1], output_dim=2),
        "class_names": [int_to_class[0], int_to_class[1]],
        "meta": {
            "input_dim": int(X_train_scaled.shape[1]),
            "n_classes": 2,
            "train_class_counts": np.bincount(y_train.to_numpy(), minlength=2).tolist(),
            "train_full_class_counts": np.bincount(y_train_full.to_numpy(), minlength=2).tolist(),
            "test_class_counts": np.bincount(y_test.to_numpy(), minlength=2).tolist(),
            "class_mapping": class_to_int
        }
    }

# ----------------------------------------------------------
# 13) NASA-CMAPSS LOADER
# ----------------------------------------------------------
def load_nasa_case(seed: int = CONFIG.seed) -> Dict[str, Any]:
    path = kagglehub.dataset_download("bishals098/nasa-turbofan-engine-degradation-simulation")

    train = pd.read_csv(os.path.join(path, "train_FD001.txt"), sep=r"\s+", header=None)

    column_names = (
        ["engine_id", "cycle", "setting_1", "setting_2", "setting_3"]
        + [f"sensor_{i}" for i in range(1, 22)]
    )
    train.columns = column_names

    constant_cols = [
        "setting_3", "sensor_1", "sensor_5", "sensor_10",
        "sensor_16", "sensor_18", "sensor_19"
    ]
    df = train.drop(columns=constant_cols).copy()

    max_cycle = df.groupby("engine_id")["cycle"].transform("max")
    df["RUL"] = max_cycle - df["cycle"]
    df["label"] = (df["RUL"] <= CONFIG.nasa_risk_horizon_rul).astype(int)

    df = df.rename(columns={
        "sensor_11": "feature1",
        "sensor_2": "feature2",
        "sensor_7": "feature3"
    })

    engine_ids = df["engine_id"].drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    engine_ids = rng.permutation(engine_ids)

    n_total = len(engine_ids)
    n_train = int(0.70 * n_total)
    n_val = int(0.15 * n_total)

    train_ids = engine_ids[:n_train]
    val_ids = engine_ids[n_train:n_train + n_val]
    test_ids = engine_ids[n_train + n_val:]

    df_train = df[df["engine_id"].isin(train_ids)].copy()
    df_val = df[df["engine_id"].isin(val_ids)].copy()
    df_train_full = df[df["engine_id"].isin(np.concatenate([train_ids, val_ids]))].copy()
    df_test = df[df["engine_id"].isin(test_ids)].copy()

    feat_cols = ["feature1", "feature2", "feature3"]

    scaler_search = MinMaxScaler()
    X_train = scaler_search.fit_transform(df_train[feat_cols]).astype(np.float32)
    X_val = scaler_search.transform(df_val[feat_cols]).astype(np.float32)

    scaler_final = MinMaxScaler()
    X_train_full = scaler_final.fit_transform(df_train_full[feat_cols]).astype(np.float32)
    X_test = scaler_final.transform(df_test[feat_cols]).astype(np.float32)

    y_train = df_train["label"].to_numpy(dtype=np.int64)
    y_val = df_val["label"].to_numpy(dtype=np.int64)
    y_train_full = df_train_full["label"].to_numpy(dtype=np.int64)
    y_test = df_test["label"].to_numpy(dtype=np.int64)

    search_counts = np.bincount(y_train, minlength=2)
    final_counts = np.bincount(y_train_full, minlength=2)

    search_class_weights = torch.tensor(
        search_counts.sum() / (len(search_counts) * search_counts),
        dtype=torch.float32
    )
    final_class_weights = torch.tensor(
        final_counts.sum() / (len(final_counts) * final_counts),
        dtype=torch.float32
    )

    train_search_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )
    train_full_ds = TensorDataset(
        torch.tensor(X_train_full, dtype=torch.float32),
        torch.tensor(y_train_full, dtype=torch.long)
    )
    test_ds = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long)
    )

    return {
        "dataset_name": "NASA_CMAPSS_FD001",
        "positive_class_name": "Risk",
        "selection_metric": "val_balanced_accuracy",
        "final_epochs": CONFIG.final_epochs_nasa,
        "train_search_loader": make_loader(train_search_ds, CONFIG.batch_size_train, True, seed=seed),
        "val_loader": make_loader(val_ds, CONFIG.batch_size_eval, False, seed=seed),
        "train_full_loader": make_loader(train_full_ds, CONFIG.batch_size_train, True, seed=seed),
        "test_loader": make_loader(test_ds, CONFIG.batch_size_eval, False, seed=seed),
        "search_class_weights": search_class_weights,
        "final_class_weights": final_class_weights,
        "model_builder": lambda: TabularMLP(input_dim=3, output_dim=2),
        "class_names": ["Healthy", "Risk"],
        "meta": {
            "n_engines_total": int(df["engine_id"].nunique()),
            "train_engines": int(len(train_ids)),
            "val_engines": int(len(val_ids)),
            "test_engines": int(len(test_ids)),
            "train_class_counts": search_counts.tolist(),
            "train_full_class_counts": final_counts.tolist(),
            "test_class_counts": np.bincount(y_test, minlength=2).tolist(),
            "positive_rate_train": float(y_train.mean()),
            "positive_rate_val": float(y_val.mean()),
            "positive_rate_test": float(y_test.mean()),
            "risk_horizon_rul": CONFIG.nasa_risk_horizon_rul
        }
    }

# ----------------------------------------------------------
# 14) GENERAL DATASET RUNNER
# ----------------------------------------------------------
def run_dataset_experiment(case: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = case["dataset_name"]
    selection_metric = case["selection_metric"]

    print(f"\n{'='*70}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*70}")
    print("Meta:", case["meta"])

    dynamic_epochs = resolve_dynamic_epochs_for_dsi(
        train_loader=case["train_search_loader"],
        min_points=CONFIG.min_series_points,
        max_batches=CONFIG.dynamic_search_max_batches
    )
    print(f"[{dataset_name}] batches_search={len(case['train_search_loader'])} | dynamic_epochs={dynamic_epochs}")

    df_dynamic = run_lr_sweep(
        model_builder=case["model_builder"],
        train_loader=case["train_search_loader"],
        learning_rates=LEARNING_RATES,
        class_weights=case["search_class_weights"],
        optimizer_name="adam",
        epochs=dynamic_epochs,
        max_batches=CONFIG.dynamic_search_max_batches,
        run_seed=CONFIG.seed
    )

    df_val = run_validation_metric_sweep(
        model_builder=case["model_builder"],
        train_loader=case["train_search_loader"],
        val_loader=case["val_loader"],
        learning_rates=LEARNING_RATES,
        class_weights=case["search_class_weights"],
        optimizer_name="adam",
        epochs=CONFIG.exploratory_epochs,
        run_seed=CONFIG.seed
    )

    df_early = run_early_loss_sweep(
        model_builder=case["model_builder"],
        train_loader=case["train_search_loader"],
        learning_rates=LEARNING_RATES,
        class_weights=case["search_class_weights"],
        optimizer_name="adam",
        run_seed=CONFIG.seed
    )

    df_random = run_random_search_validation(
        model_builder=case["model_builder"],
        train_loader=case["train_search_loader"],
        val_loader=case["val_loader"],
        learning_rates=LEARNING_RATES,
        class_weights=case["search_class_weights"],
        optimizer_name="adam",
        epochs=CONFIG.exploratory_epochs,
        n_samples=CONFIG.random_search_samples,
        sampling_seed=CONFIG.seed,
        run_seed=CONFIG.seed
    )

    grid_lr = select_lr_by_metric(df_val, selection_metric)
    dynamic_lr, dynamic_configs = dynamic_rule_quantile_plus_descent(df_dynamic, quantile=0.33)
    early_lr = select_lr_by_early_loss_sweep(df_early)
    random_lr = select_lr_by_metric(df_random, selection_metric)

    print("\nSelected learning rates:")
    print("Grid:", grid_lr)
    print("Dynamic:", dynamic_lr)
    print("Early-loss:", early_lr)
    print("Random:", random_lr)

    grid_search_time = df_val["elapsed_seconds"].sum()
    dynamic_search_time = df_dynamic["elapsed_seconds"].sum()
    early_search_time = df_early["elapsed_seconds"].sum()
    random_search_time = df_random["elapsed_seconds"].sum()

    grid_search_batches = df_val["batches_processed"].sum()
    dynamic_search_batches = df_dynamic["batches_processed"].sum()
    early_search_batches = df_early["batches_processed"].sum()
    random_search_batches = df_random["batches_processed"].sum()

    method_specs = [
        ("Grid Search", grid_lr, grid_search_time, grid_search_batches),
        ("Dynamic Criterion", dynamic_lr, dynamic_search_time, dynamic_search_batches),
        ("Early-loss Sweep", early_lr, early_search_time, early_search_batches),
        ("Random Search", random_lr, random_search_time, random_search_batches),
    ]

    comparison_rows = []
    final_results_by_method = {}

    for method_name, chosen_lr, stime, sbatches in method_specs:
        final_out = train_final_and_evaluate(
            model_builder=case["model_builder"],
            train_full_loader=case["train_full_loader"],
            test_loader=case["test_loader"],
            lr=float(chosen_lr),
            class_weights=case["final_class_weights"],
            epochs=case["final_epochs"],
            run_seed=CONFIG.seed
        )

        final_results_by_method[method_name] = final_out

        comparison_rows.append({
            "dataset": dataset_name,
            "method": method_name,
            "learning_rate": float(chosen_lr),
            "test_accuracy": final_out["test_accuracy"],
            "test_balanced_accuracy": final_out["test_balanced_accuracy"],
            "test_precision_class_1": final_out["test_precision_class_1"],
            "test_recall_class_1": final_out["test_recall_class_1"],
            "test_f1_class_1": final_out["test_f1_class_1"],
            "search_time_seconds": stime,
            "search_batches": sbatches,
            "final_train_time_seconds": final_out["train_time_seconds"],
            "final_train_batches": final_out["batches_processed"],
            "final_diverged": final_out["diverged"]
        })

    df_comparison = pd.DataFrame(comparison_rows)
    df_comparison["accuracy_per_search_second"] = df_comparison.apply(
        lambda row: safe_divide(row["test_accuracy"], row["search_time_seconds"]),
        axis=1
    )
    df_comparison["balanced_accuracy_per_search_second"] = df_comparison.apply(
        lambda row: safe_divide(row["test_balanced_accuracy"], row["search_time_seconds"]),
        axis=1
    )
    df_comparison["accuracy_per_100_batches"] = df_comparison.apply(
        lambda row: safe_divide(row["test_accuracy"], row["search_batches"] / 100.0),
        axis=1
    )
    df_comparison["balanced_accuracy_per_100_batches"] = df_comparison.apply(
        lambda row: safe_divide(row["test_balanced_accuracy"], row["search_batches"] / 100.0),
        axis=1
    )

    dynamic_final = final_results_by_method["Dynamic Criterion"]
    y_true = dynamic_final["y_true"]
    y_pred = dynamic_final["y_pred"]

    cm = confusion_matrix(y_true, y_pred)
    df_confusion = pd.DataFrame(
        cm,
        index=[f"Actual {c}" for c in case["class_names"]],
        columns=[f"Predicted {c}" for c in case["class_names"]]
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=case["class_names"],
        output_dict=True,
        zero_division=0
    )
    df_report = (
        pd.DataFrame(report_dict)
        .transpose()
        .reset_index()
        .rename(columns={"index": "label"})
    )

    df_metrics_dynamic = pd.DataFrame([
        {"metric": "accuracy", "value": dynamic_final["test_accuracy"]},
        {"metric": "balanced_accuracy", "value": dynamic_final["test_balanced_accuracy"]},
        {"metric": "precision_class_1", "value": dynamic_final["test_precision_class_1"]},
        {"metric": "recall_class_1", "value": dynamic_final["test_recall_class_1"]},
        {"metric": "f1_class_1", "value": dynamic_final["test_f1_class_1"]},
        {"metric": "dynamic_lr_used", "value": dynamic_lr},
        {"metric": "dynamic_search_time_seconds", "value": dynamic_search_time},
        {"metric": "dynamic_search_batches", "value": dynamic_search_batches}
    ])

    summary_row = {
        "dataset": dataset_name,
        "positive_class": case["positive_class_name"],
        "selection_metric": selection_metric,
        "grid_lr": grid_lr,
        "dynamic_lr": dynamic_lr,
        "early_lr": early_lr,
        "random_lr": random_lr,
        "grid_test_accuracy": df_comparison.loc[df_comparison["method"] == "Grid Search", "test_accuracy"].iloc[0],
        "dynamic_test_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_accuracy"].iloc[0],
        "early_test_accuracy": df_comparison.loc[df_comparison["method"] == "Early-loss Sweep", "test_accuracy"].iloc[0],
        "random_test_accuracy": df_comparison.loc[df_comparison["method"] == "Random Search", "test_accuracy"].iloc[0],
        "grid_test_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Grid Search", "test_balanced_accuracy"].iloc[0],
        "dynamic_test_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_balanced_accuracy"].iloc[0],
        "early_test_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Early-loss Sweep", "test_balanced_accuracy"].iloc[0],
        "random_test_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Random Search", "test_balanced_accuracy"].iloc[0],
        "dynamic_minus_grid_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_accuracy"].iloc[0] - df_comparison.loc[df_comparison["method"] == "Grid Search", "test_accuracy"].iloc[0],
        "dynamic_minus_grid_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_balanced_accuracy"].iloc[0] - df_comparison.loc[df_comparison["method"] == "Grid Search", "test_balanced_accuracy"].iloc[0],
        "dynamic_minus_random_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_accuracy"].iloc[0] - df_comparison.loc[df_comparison["method"] == "Random Search", "test_accuracy"].iloc[0],
        "dynamic_minus_random_balanced_accuracy": df_comparison.loc[df_comparison["method"] == "Dynamic Criterion", "test_balanced_accuracy"].iloc[0] - df_comparison.loc[df_comparison["method"] == "Random Search", "test_balanced_accuracy"].iloc[0],
        "grid_search_time_seconds": grid_search_time,
        "dynamic_search_time_seconds": dynamic_search_time,
        "early_search_time_seconds": early_search_time,
        "random_search_time_seconds": random_search_time,
        "grid_search_batches": grid_search_batches,
        "dynamic_search_batches": dynamic_search_batches,
        "early_search_batches": early_search_batches,
        "random_search_batches": random_search_batches,
        "time_speedup_grid_over_dynamic": safe_divide(grid_search_time, dynamic_search_time),
        "batch_speedup_grid_over_dynamic": safe_divide(grid_search_batches, dynamic_search_batches),
        "dynamic_precision_class_1": dynamic_final["test_precision_class_1"],
        "dynamic_recall_class_1": dynamic_final["test_recall_class_1"],
        "dynamic_f1_class_1": dynamic_final["test_f1_class_1"],
        "dynamic_tn": int(cm[0, 0]),
        "dynamic_fp": int(cm[0, 1]),
        "dynamic_fn": int(cm[1, 0]),
        "dynamic_tp": int(cm[1, 1]),
    }

    return {
        "dataset_name": dataset_name,
        "comparison": df_comparison,
        "summary": pd.DataFrame([summary_row]),
        "dynamic_metrics": df_metrics_dynamic,
        "dynamic_confusion": df_confusion,
        "dynamic_report": df_report,
        "dynamic_predictions": dynamic_final["predictions_df"],
        "dynamic_cm_array": cm,
        "dynamic_search_table": df_dynamic,
        "validation_search_table": df_val,
        "early_loss_table": df_early,
        "random_search_table": df_random
    }

# ----------------------------------------------------------
# 15) RUN RAISIN AND NASA
# ----------------------------------------------------------
raisin_case = load_raisin_case(seed=CONFIG.seed)
nasa_case = load_nasa_case(seed=CONFIG.seed)

results_raisin = run_dataset_experiment(raisin_case)
results_nasa = run_dataset_experiment(nasa_case)

# ----------------------------------------------------------
# 16) SHOW INDIVIDUAL RESULTS
# ----------------------------------------------------------
print("\n=== INDIVIDUAL RESULTS: RAISIN ===\n")
display(results_raisin["comparison"])
display(results_raisin["summary"])
display(results_raisin["dynamic_metrics"])
display(results_raisin["dynamic_confusion"])
display(results_raisin["dynamic_report"])

print("\n=== INDIVIDUAL RESULTS: NASA-CMAPSS ===\n")
display(results_nasa["comparison"])
display(results_nasa["summary"])
display(results_nasa["dynamic_metrics"])
display(results_nasa["dynamic_confusion"])
display(results_nasa["dynamic_report"])

# ----------------------------------------------------------
# 17) FINAL COMPARATIVE TABLE
# ----------------------------------------------------------
df_final_comparative = pd.concat(
    [results_raisin["summary"], results_nasa["summary"]],
    axis=0,
    ignore_index=True
)

df_final_comparative_compact = df_final_comparative[[
    "dataset",
    "positive_class",
    "selection_metric",
    "grid_lr",
    "dynamic_lr",
    "early_lr",
    "random_lr",
    "grid_test_accuracy",
    "dynamic_test_accuracy",
    "grid_test_balanced_accuracy",
    "dynamic_test_balanced_accuracy",
    "dynamic_minus_grid_accuracy",
    "dynamic_minus_grid_balanced_accuracy",
    "time_speedup_grid_over_dynamic",
    "batch_speedup_grid_over_dynamic",
    "dynamic_precision_class_1",
    "dynamic_recall_class_1",
    "dynamic_f1_class_1",
    "dynamic_tn",
    "dynamic_fp",
    "dynamic_fn",
    "dynamic_tp"
]].copy()

print("\n=== FINAL COMPARATIVE TABLE: RAISIN VS NASA ===\n")
display(df_final_comparative_compact)

# ----------------------------------------------------------
# 18) PLOTLY CONFUSION MATRICES
# ----------------------------------------------------------
fig_raisin_cm = px.imshow(
    results_raisin["dynamic_confusion"].values,
    x=results_raisin["dynamic_confusion"].columns,
    y=results_raisin["dynamic_confusion"].index,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Raisin — Confusion Matrix (Dynamic Criterion)"
)
fig_raisin_cm.update_layout(
    template="plotly_dark",
    title_x=0.5,
    xaxis_title="Predicted class",
    yaxis_title="Actual class",
    coloraxis_colorbar_title="Frequency"
)
fig_raisin_cm.update_xaxes(side="bottom")
fig_raisin_cm.show()

fig_nasa_cm = px.imshow(
    results_nasa["dynamic_confusion"].values,
    x=results_nasa["dynamic_confusion"].columns,
    y=results_nasa["dynamic_confusion"].index,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="NASA-CMAPSS FD001 — Confusion Matrix (Dynamic Criterion)"
)
fig_nasa_cm.update_layout(
    template="plotly_dark",
    title_x=0.5,
    xaxis_title="Predicted class",
    yaxis_title="Actual class",
    coloraxis_colorbar_title="Frequency"
)
fig_nasa_cm.update_xaxes(side="bottom")
fig_nasa_cm.show()

# ----------------------------------------------------------
# 19) SAVE ARTIFACTS
# ----------------------------------------------------------
with open(os.path.join(OUTPUT_DIR, "real_world_dsi_config.json"), "w", encoding="utf-8") as f:
    json.dump(asdict(CONFIG), f, indent=4)

# Raisin
results_raisin["comparison"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_comparison.csv"),
    index=False
)
results_raisin["summary"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_summary.csv"),
    index=False
)
results_raisin["dynamic_metrics"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_dynamic_metrics.csv"),
    index=False
)
results_raisin["dynamic_confusion"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_dynamic_confusion.csv"),
    index=True
)
results_raisin["dynamic_report"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_dynamic_classification_report.csv"),
    index=False
)
results_raisin["dynamic_predictions"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_dynamic_predictions.csv"),
    index=False
)
results_raisin["dynamic_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_dynamic_search_table.csv"),
    index=False
)
results_raisin["validation_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_validation_search_table.csv"),
    index=False
)
results_raisin["early_loss_table"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_early_loss_search_table.csv"),
    index=False
)
results_raisin["random_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "raisin_random_search_table.csv"),
    index=False
)

# NASA
results_nasa["comparison"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_comparison.csv"),
    index=False
)
results_nasa["summary"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_summary.csv"),
    index=False
)
results_nasa["dynamic_metrics"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_metrics.csv"),
    index=False
)
results_nasa["dynamic_confusion"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_confusion.csv"),
    index=True
)
results_nasa["dynamic_report"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_classification_report.csv"),
    index=False
)
results_nasa["dynamic_predictions"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_predictions.csv"),
    index=False
)
results_nasa["dynamic_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_search_table.csv"),
    index=False
)
results_nasa["validation_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_validation_search_table.csv"),
    index=False
)
results_nasa["early_loss_table"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_early_loss_search_table.csv"),
    index=False
)
results_nasa["random_search_table"].to_csv(
    os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_random_search_table.csv"),
    index=False
)

# Final comparison
df_final_comparative.to_csv(
    os.path.join(OUTPUT_DIR, "real_world_final_comparison_full.csv"),
    index=False
)
df_final_comparative_compact.to_csv(
    os.path.join(OUTPUT_DIR, "real_world_final_comparison_compact.csv"),
    index=False
)

export_dataframe_html(
    df_final_comparative,
    "Real-world applications — Full comparison",
    os.path.join(OUTPUT_DIR, "real_world_final_comparison_full.html")
)
export_dataframe_html(
    df_final_comparative_compact,
    "Real-world applications — Compact comparison",
    os.path.join(OUTPUT_DIR, "real_world_final_comparison_compact.html")
)

# Confusion matrix HTML
raisin_cm_html = os.path.join(OUTPUT_DIR, "raisin_dynamic_confusion_matrix.html")
nasa_cm_html = os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_confusion_matrix.html")
fig_raisin_cm.write_html(raisin_cm_html)
fig_nasa_cm.write_html(nasa_cm_html)

print("\nSaved files:")
print(os.path.join(OUTPUT_DIR, "raisin_comparison.csv"))
print(os.path.join(OUTPUT_DIR, "raisin_summary.csv"))
print(os.path.join(OUTPUT_DIR, "raisin_dynamic_metrics.csv"))
print(os.path.join(OUTPUT_DIR, "raisin_dynamic_confusion.csv"))
print(os.path.join(OUTPUT_DIR, "raisin_dynamic_classification_report.csv"))
print(os.path.join(OUTPUT_DIR, "raisin_dynamic_predictions.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_comparison.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_summary.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_metrics.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_confusion.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_classification_report.csv"))
print(os.path.join(OUTPUT_DIR, "nasa_cmapss_fd001_dynamic_predictions.csv"))
print(os.path.join(OUTPUT_DIR, "real_world_final_comparison_full.csv"))
print(os.path.join(OUTPUT_DIR, "real_world_final_comparison_compact.csv"))
print(os.path.join(OUTPUT_DIR, "real_world_final_comparison_full.html"))
print(os.path.join(OUTPUT_DIR, "real_world_final_comparison_compact.html"))
print(raisin_cm_html)
print(nasa_cm_html)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTPUT_DIR = /content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs
DATA_DIR   = /content/drive/MyDrive/dynamic_stability_lr_project/data_cache
DEVICE     = cuda
Learning rates = [1.00000000e-04 2.78255940e-04 7.74263683e-04 2.15443469e-03
 5.99484250e-03 1.66810054e-02 4.64158883e-02 1.29154967e-01
 3.59381366e-01 1.00000000e+00]
Using Colab cache for faster access to the 'nasa-turbofan-engine-degradation-simulation' dataset.

DATASET: Raisin
Meta: {'input_dim': 7, 'n_classes': 2, 'train_class_counts': [315, 315], 'train_full_class_counts': [382, 383], 'test_class_counts': [68, 67], 'class_mapping': {'Besni': 0, 'Kecimen': 1}}
[Raisin] batches_search=10 | dynamic_epochs=1

Selected learning rates:
Grid: 0.046415888336127774
Dynamic: 0.016681005372000592
Early-loss: 0.046415888336127774
Random: 0.046415888336127774

DATASET: NASA_CMAPSS_FD001
Meta:

,dataset,method,learning_rate,test_accuracy,test_balanced_accuracy,test_precision_class_1,test_recall_class_1,test_f1_class_1,search_time_seconds,search_batches,final_train_time_seconds,final_train_batches,final_diverged,accuracy_per_search_second,balanced_accuracy_per_search_second,accuracy_per_100_batches,balanced_accuracy_per_100_batches
0,Raisin,Grid Search,0.046416,0.881481,0.882243,0.814815,0.985075,0.891892,4.537867,300,2.752061,240,False,0.194250,0.194418,0.293827,0.294081
1,Raisin,Dynamic Criterion,0.016681,0.881481,0.882133,0.822785,0.970149,0.890411,1.938973,100,4.065326,240,False,0.454613,0.454949,0.881481,0.882133
2,Raisin,Early-loss Sweep,0.046416,0.881481,0.882243,0.814815,0.985075,0.891892,1.439447,100,2.820807,240,False,0.612375,0.612904,0.881481,0.882243
3,Raisin,Random Search,0.046416,0.881481,0.882243,0.814815,0.985075,0.891892,1.568269,120,2.735499,240,False,0.562073,0.562559,0.734568,0.735203


,dataset,positive_class,selection_metric,grid_lr,dynamic_lr,early_lr,random_lr,grid_test_accuracy,dynamic_test_accuracy,early_test_accuracy,...,random_search_batches,time_speedup_grid_over_dynamic,batch_speedup_grid_over_dynamic,dynamic_precision_class_1,dynamic_recall_class_1,dynamic_f1_class_1,dynamic_tn,dynamic_fp,dynamic_fn,dynamic_tp
0,Raisin,Kecimen,val_accuracy,0.046416,0.016681,0.046416,0.046416,0.881481,0.881481,0.881481,...,120,2.340346,3.0,0.822785,0.970149,0.890411,54,14,2,65


,metric,value
0,accuracy,0.881481
1,balanced_accuracy,0.882133
2,precision_class_1,0.822785
3,recall_class_1,0.970149
4,f1_class_1,0.890411
5,dynamic_lr_used,0.016681
6,dynamic_search_time_seconds,1.938973
7,dynamic_search_batches,100.000000


,Predicted Besni,Predicted Kecimen
Actual Besni,54,14
Actual Kecimen,2,65


...

,Predicted Healthy,Predicted Risk
Actual Healthy,2459,140
Actual Risk,70,395


,label,precision,recall,f1-score,support
0,Healthy,0.972321,0.946133,0.959048,2599.000000
1,Risk,0.738318,0.849462,0.790000,465.000000
2,accuracy,0.931462,0.931462,0.931462,0.931462
3,macro avg,0.855319,0.897798,0.874524,3064.000000
4,weighted avg,0.936808,0.931462,0.933393,3064.000000



=== FINAL COMPARATIVE TABLE: RAISIN VS NASA ===



,dataset,positive_class,selection_metric,grid_lr,dynamic_lr,early_lr,random_lr,grid_test_accuracy,dynamic_test_accuracy,grid_test_balanced_accuracy,...,dynamic_minus_grid_balanced_accuracy,time_speedup_grid_over_dynamic,batch_speedup_grid_over_dynamic,dynamic_precision_class_1,dynamic_recall_class_1,dynamic_f1_class_1,dynamic_tn,dynamic_fp,dynamic_fn,dynamic_tp
0,Raisin,Kecimen,val_accuracy,0.046416,0.016681,0.046416,0.046416,0.881481,0.881481,0.882243,...,-0.000110,2.340346,3.0,0.822785,0.970149,0.890411,54,14,2,65
1,NASA_CMAPSS_FD001,Risk,val_balanced_accuracy,0.000278,0.046416,0.046416,0.046416,0.904047,0.931462,0.907241,...,-0.009444,14.492814,22.6,0.738318,0.849462,0.790000,2459,140,70,395



Saved files:
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_comparison.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_summary.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_dynamic_metrics.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_dynamic_confusion.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_dynamic_classification_report.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/raisin_dynamic_predictions.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/nasa_cmapss_fd001_comparison.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/nasa_cmapss_fd001_summary.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/nasa_cmapss_fd001_dynamic_metrics.csv
/content/drive/MyDrive/dynamic_stability_lr_project/paper_outputs/nasa_cmapss_fd001_dynamic_confusion.csv
/content/drive/M